# Cloud Chaser Kaggle Notebook

This notebook recreates the full Cloud Chaser codebase inside a Kaggle working directory. Segmentation uses the Kaggle Sky-Image Dataset `swimseg-2` cloud masks, while cloud-type classification keeps GCD. Set Kaggle accelerator to GPU before running training cells.

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/kaggle/working/cloud-chaser')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

for directory in ['cloud_chaser', 'cloud_chaser/data', 'cloud_chaser/models', 'cloud_chaser/utils', 'configs', 'scripts']:
    Path(directory).mkdir(parents=True, exist_ok=True)

print('Working in', PROJECT_DIR)


## Write Project Files

These cells materialize the Python package and CLI files.

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "cloud-chaser"
version = "0.1.0"
description = "Two-stage cloud detection, segmentation, and meteorological classification."
readme = "README.md"
requires-python = ">=3.10,<3.13"
dependencies = [
  "albumentations>=1.4.0",
  "numpy>=1.24",
  "opencv-python>=4.8",
  "pillow>=10.0",
  "pyyaml>=6.0",
  "scikit-learn>=1.3",
  "torch>=2.1",
  "torchvision>=0.16",
  "tqdm>=4.66",
  "ultralytics>=8.2.0",
]

[project.optional-dependencies]
dev = ["ruff>=0.5", "pytest>=8.0"]

[project.scripts]
cloud-chaser-train = "train:main"
cloud-chaser-infer = "inference:main"
cloud-chaser-export = "export:main"

[tool.setuptools.packages.find]
include = ["cloud_chaser*"]

[tool.setuptools]
py-modules = ["train", "inference", "export"]

[tool.ruff]
line-length = 100
target-version = "py310"


In [ ]:
%%writefile requirements.txt
albumentations>=1.4.0
matplotlib>=3.8
numpy>=1.24
opencv-python>=4.8
pillow>=10.0
pyyaml>=6.0
scikit-learn>=1.3
torch>=2.1
torchvision>=0.16
tqdm>=4.66
ultralytics>=8.2.0


In [ ]:
%%writefile README.md
# Cloud Chaser

## Abstract

Cloud Chaser is a two-stage computer vision system for cloud identification in unconstrained outdoor imagery. The system first localizes cloud-like regions using an instance segmentation model trained on SWIMSEG cloud masks, then classifies each detected cloud region into a meteorological category using a deep convolutional classifier. The design separates geometric localization from cloud-type recognition, allowing the segmentation module to learn cloud boundaries while the classification module focuses on texture, shape, and spatial structure within extracted cloud regions.

## Local Quick Start

Expected local datasets:

```text
data/swimseg-2/**              # SWIMSEG/SkyImage image-mask pairs for cloud segmentation
data/GCD/train/<class>/*.jpg   # 7-class cloud classifier data
data/GCD/test/<class>/*.jpg
```

Prepare SWIMSEG masks for YOLO-Seg:

```bash
python -m scripts.prepare_swimseg_yolo --config configs/default.yaml
```

Train/evaluate the detector:

```bash
python train.py detector --config configs/default.yaml
python train.py eval-detector --config configs/default.yaml
```

Train the U-Net fallback detector and the classifier:

```bash
python train.py unet --config configs/default.yaml
python train.py eval-unet --config configs/default.yaml
python train.py classifier --config configs/default.yaml
python train.py eval-classifier --config configs/default.yaml
```

## 1. Problem Definition

The objective is to process a raw outdoor image and produce pixel-level cloud masks, bounding boxes, confidence scores, and meteorological cloud-type labels. Formally, given an image \(I \in \mathbb{R}^{H \times W \times 3}\), the system predicts a set of cloud instances:

\[
\mathcal{P} = \{(M_i, B_i, c_i, s_i, p_i)\}_{i=1}^{N}
\]

where \(M_i\) is a binary segmentation mask, \(B_i\) is a bounding box, \(c_i\) is the predicted cloud type, \(s_i\) is the segmentation confidence, and \(p_i\) is the classification confidence.

The task is challenging because clouds have ambiguous boundaries, variable scale, high intra-class diversity, and frequent visual overlap with sky, haze, smoke, snow, bright buildings, and other outdoor structures.

## 2. System Architecture

Cloud Chaser is implemented as a two-stage cascade rather than a single whole-image classifier. The detector is responsible for deciding where cloud material exists, and the classifier is responsible for assigning a meteorological class only to the regions that pass the detector gate.

```text
Raw image
  |
  v
Hybrid cloud detector
  |-- YOLO-Seg primary detector
  |-- U-Net fallback detector
  |-- cloud mask(s)
  |-- bounding box(es)
  |-- detection confidence(s)
  |
  v
Mask-guided crop extraction
  |
  v
CNN cloud-type classifier
  |
  v
Annotated image + structured predictions
```

This separation matters because cloud localization and cloud-type recognition are different problems. Localization depends on pixel-level shape, contrast, and cloud-vs-sky boundaries. Classification depends more on texture, density, vertical development, cloud layering, and spatial structure. Keeping the two responsibilities separate makes it possible to evaluate detector failure and classifier failure independently.

### 2.1 Detector Stage

The detector stage uses a hybrid design. YOLO-Seg is the primary detector, configured by default as `yolo11s-seg.pt`. It is fine-tuned on SWIMSEG cloud masks converted into YOLO polygon labels. The model is a one-class instance segmentation detector whose foreground class is `cloud`.

U-Net is the fallback detector. It is trained on the same SWIMSEG masks, but it predicts a dense binary cloud probability map instead of YOLO-style instances. During inference, if YOLO returns no cloud detections, the U-Net branch runs over the full image. Its probability map is thresholded, small components are removed, and connected components are converted into masks, boxes, and confidence scores.

For each detected cloud region, the detector backend returns:

- `box`: an axis-aligned bounding box in original image coordinates,
- `mask`: a binary cloud mask,
- `confidence`: YOLO confidence for YOLO detections, or mean U-Net probability inside the connected component for fallback detections.

The detector is intentionally not asked to classify cloud type. Its job is to answer: "Is there cloud material here, and which pixels belong to it?"

### 2.2 Crop Extraction Stage

The crop extraction stage converts detector outputs into classifier inputs. For every detected instance:

1. The predicted mask is resized back to the original image resolution if needed.
2. The bounding box is padded by `inference.crop_padding`.
3. Pixels outside the predicted mask are zeroed.
4. The resulting crop is resized to `data.image_size`.
5. ImageNet normalization is applied before batching.

Mask-guided cropping reduces background leakage. A classifier trained on whole GCD images can learn shortcuts from sky color, horizon position, or exposure. The cascade tries to force the classifier to focus on the detected cloud region.

### 2.3 Classifier Stage

The classifier stage is a PyTorch CNN with a configurable backbone:

- `resnet50`,
- `efficientnet_b0`,
- `densenet121`.

The default is ResNet50. The backbone acts as a feature encoder, followed by a dropout layer and a linear classification head over the seven GCD classes:

```text
masked cloud crop -> CNN encoder -> feature vector -> dropout -> linear head -> logits
```

At inference time, logits are converted with softmax. The top class and probability are attached to the corresponding detector instance. If the detector returns multiple cloud instances, the classifier processes all crops as a batch for efficiency.

### 2.4 Training Source Boundaries

GCD is treated as the supervised classification dataset, not as an unlabeled pretraining source. The classifier therefore uses an ImageNet-pretrained backbone followed by supervised GCD fine-tuning. This avoids mixing the classification labels into an unrelated pretraining stage and keeps the training flow simpler.

### 2.5 Inference Cascade

The production inference path is:

1. Read a raw image with OpenCV.
2. Run YOLO-Seg with configured confidence and IoU thresholds.
3. If YOLO detects clouds, use YOLO masks and boxes.
4. If YOLO detects nothing and `detector.backend: hybrid`, run U-Net as a fallback.
5. Convert the U-Net probability map into connected cloud components.
6. For each detected mask and box, create a mask-guided crop.
7. Batch all crops and run the classifier.
8. Overlay masks, boxes, class labels, and confidence scores.

This means a high standalone classifier score does not guarantee a high end-to-end pipeline score. If the detector misses the cloud, the classifier never receives a crop. For this reason, the project reports both standalone classifier metrics and cascade metrics.

### 2.6 Main Artifacts

Training and export produce the following primary artifacts:

```text
runs/detector/weights/best.pt       # YOLO-Seg detector checkpoint
runs/detector/weights/best.onnx     # optional detector ONNX export
runs/unet/best.pt                   # U-Net semantic segmentation fallback checkpoint
runs/classifier/best.pt             # PyTorch classifier checkpoint with metadata
classifier.torchscript              # optional standalone classifier export
```

The normal Python inference pipeline can load `runs/classifier/best.pt`. It can also load `classifier.torchscript` when class names are supplied from `configs/default.yaml`.

## 3. Dataset Strategy

### 3.1 Localization Dataset: SWIMSEG / SkyImage

SWIMSEG from the SkyImage dataset is used to train the segmentation component because it provides explicit cloud masks. This is a better segmentation target than generic scene labels: the detector learns cloud-vs-sky boundaries directly instead of relying on `sky` as a proxy foreground class.

The local converter discovers image-mask pairs, binarizes the cloud masks, extracts connected components, converts them into YOLO-Seg polygons, and writes a one-class segmentation dataset where the foreground class is `cloud`.

### 3.2 Classification Dataset: GCD

The GCD dataset provides labeled images for seven cloud categories:

1. Cumulus
2. Altocumulus
3. Cirrus
4. Clear sky
5. Stratocumulus
6. Cumulonimbus
7. Mixed cloud

The classifier is trained using standard train, validation, and test partitions. If no validation directory exists, a stratified validation subset is deterministically sampled from the training set.

## 4. Data Preprocessing and Augmentation

### 4.1 SWIMSEG Mask Conversion

SWIMSEG binary masks are converted into YOLO segmentation labels. For each mask, foreground cloud pixels are extracted, connected components are filtered by minimum area, converted into polygon contours, normalized to image coordinates, and written in YOLO segmentation format.

This conversion enables fine-tuning YOLO-Seg models using pixel-level supervision while preserving instance-like connected regions.

### 4.2 Meteorology-Aware Augmentations

The classification and self-supervised pipelines use augmentations selected to preserve meteorological texture patterns while improving robustness:

- Random shadows simulate illumination changes and partial occlusions.
- Gaussian blur models atmospheric haze, lens softness, and motion blur.
- Horizontal flips preserve cloud texture statistics and increase viewpoint diversity.
- Conservative vertical flips are included because cloud texture is often orientation-tolerant, but the probability is kept lower than horizontal flips to avoid excessive physical distortion.
- Color jitter accounts for exposure, white balance, and time-of-day variation.

## 5. Segmentation Module

### 5.1 Architecture

The segmentation module uses two backends:

- YOLO-Seg for primary instance segmentation,
- U-Net for dense semantic segmentation fallback.

YOLO-Seg is configured by default as `yolo11s-seg.pt`. The model may also be switched to `yolov8s-seg.pt` through the YAML configuration.

YOLO-Seg is used because it provides:

- real-time or near-real-time detection speed,
- bounding boxes and masks from a single model,
- strong transfer learning from pretrained weights,
- straightforward export to ONNX and other deployment formats.

U-Net is used because cloud boundaries are often diffuse and non-object-like. It predicts a dense probability for every pixel and can recover full-frame or low-contrast cloud fields that YOLO may miss.

### 5.2 Training Objective

The YOLO detector is fine-tuned on converted SWIMSEG cloud polygons. The U-Net detector is trained directly on resized SWIMSEG binary masks using a BCE-plus-Dice objective. Both learn the same binary concept, but with different inductive biases: YOLO learns object-like cloud regions, while U-Net learns dense foreground probability.

### 5.3 Output

For each detected instance, the detector returns:

- a bounding box \(B_i = (x_1, y_1, x_2, y_2)\),
- a pixel mask \(M_i\),
- a detection confidence \(s_i\).

These outputs are passed to the classification stage.

## 6. Feature Extraction and Classification Module

### 6.1 Two-Stage Recognition

The system uses a two-stage recognition strategy:

1. Localize candidate cloud regions with YOLO-Seg.
2. Classify each detected cloud crop using a CNN classifier.

This design is preferred over whole-image classification because a general outdoor image may contain many irrelevant objects. Mask-guided cropping reduces background bias and encourages the classifier to focus on cloud morphology.

### 6.2 Backbone Choices

The classifier supports the following convolutional backbones:

- ResNet50
- EfficientNet-B0
- DenseNet121

ResNet50 is the default because it offers a strong balance between representational capacity, training stability, and deployment compatibility.

### 6.3 Supervised Fine-Tuning

The classification head is trained with cross-entropy loss over the seven GCD classes. The training pipeline supports optional backbone freezing for early epochs, which stabilizes fine-tuning from the ImageNet-pretrained encoder before the full network is unfrozen.

## 7. Inference Pipeline

At inference time, the system processes a high-resolution image through the following steps:

1. Read the raw RGB image.
2. Run YOLO-Seg to detect cloud instances.
3. Resize masks to the original image resolution when necessary.
4. Use each mask and bounding box to extract a cloud-focused crop.
5. Batch all crops and pass them through the classifier.
6. Convert logits into class probabilities using softmax.
7. Overlay masks, bounding boxes, class names, and confidence scores on the original image.

The final output is an annotated image and a structured list of predictions.

## 8. Optimization and Deployment

The implementation supports several production-oriented optimizations:

- mixed precision training and inference on CUDA,
- batched classification of detected crops,
- configurable YOLO confidence and IoU thresholds,
- ONNX export for detector and classifier,
- TorchScript export for classifier deployment,
- centralized YAML configuration for reproducible experiments.

The modular code structure separates data processing, model definitions, training logic, evaluation, inference, visualization, and export.

## 9. Evaluation Metrics

### 9.1 Segmentation Metrics

The segmentation module is evaluated using:

- mask mAP, reported by Ultralytics validation,
- mask mAP@50,
- mean Intersection over Union over SWIMSEG validation foreground masks.

For binary mIoU, the predicted foreground mask is compared with the target cloud mask:

\[
\text{IoU} = \frac{|M_{pred} \cap M_{gt}|}{|M_{pred} \cup M_{gt}|}
\]

### 9.2 Classification Metrics

The classifier is evaluated using:

- Top-1 accuracy,
- macro F1-score.

Macro F1 is important because cloud datasets may be class-imbalanced, and average accuracy alone can obscure weak performance on rarer cloud types such as cumulonimbus.

### 9.3 Cascade Metrics on GCD

The GCD dataset provides image-level category labels but does not provide cloud segmentation masks. Therefore, detector evaluation on GCD is reported as an image-level cascade proxy:

- non-clearsky classes should produce at least one cloud detection,
- `clearsky` images should produce no cloud detection,
- classification accuracy is computed only for images that pass the detector stage,
- end-to-end cascade accuracy counts an image as correct only when the detector gate and class decision are both correct.

The report script `scripts/gcd_visual_report.py` writes:

```text
reports/gcd_val_cascade_bar.png
reports/gcd_val_cascade_overlay_samples.jpg
reports/gcd_val_cascade_metrics.json
```

This distinction is important. A classifier can score highly on full GCD images while the real two-stage pipeline performs poorly if the detector misses clouds and no crop is sent to the classifier.

## 10. Reproducibility

The pipeline uses deterministic train/validation splitting for GCD and a centralized configuration file. Important experimental parameters are stored in `configs/default.yaml`, including dataset roots, image size, model backbones, learning rates, batch sizes, epoch counts, augmentation probabilities, detector thresholds, and checkpoint paths.

## 11. Limitations

The current SWIMSEG supervision is much more cloud-specific than ADE20K, but it is still primarily sky-image oriented. It may not cover every cluttered ground-level outdoor condition, such as clouds partly occluded by buildings, trees, cables, or mountains.

The classification model also depends on the quality and representativeness of GCD labels. Ambiguous cloud scenes, transitional cloud forms, and multi-layer atmospheres may be difficult to assign to a single class.

## 12. Conclusion

Cloud Chaser implements a practical two-stage approach for cloud identification in general outdoor imagery. By combining SWIMSEG-based cloud-mask segmentation, optional self-supervised cloud representation learning, and supervised GCD classification, the system balances localization robustness with meteorological specificity. The resulting pipeline is suitable for experimentation, deployment-oriented optimization, and further research into cloud-type recognition under real-world visual conditions.


In [ ]:
%%writefile train.py
from __future__ import annotations

import argparse

from cloud_chaser.config import load_config
from cloud_chaser.evaluation import evaluate_detector
from cloud_chaser.training import (
    evaluate_classifier,
    evaluate_unet_detector,
    train_classifier,
    train_detector,
    train_unet_detector,
)


def main() -> None:
    parser = argparse.ArgumentParser(description="Cloud Chaser training entrypoint")
    parser.add_argument(
        "stage",
        choices=["detector", "unet", "classifier", "eval-classifier", "eval-detector", "eval-unet"],
    )
    parser.add_argument("--config", default="configs/default.yaml")
    args = parser.parse_args()

    cfg = load_config(args.config)
    if args.stage == "detector":
        train_detector(cfg)
    elif args.stage == "unet":
        train_unet_detector(cfg)
    elif args.stage == "classifier":
        train_classifier(cfg)
    elif args.stage == "eval-classifier":
        evaluate_classifier(cfg)
    elif args.stage == "eval-detector":
        evaluate_detector(cfg)
    elif args.stage == "eval-unet":
        evaluate_unet_detector(cfg)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile inference.py
from __future__ import annotations

import argparse
from pathlib import Path

import cv2

from cloud_chaser.config import get_device, load_config
from cloud_chaser.inference_pipeline import CloudIdentifier


def main() -> None:
    parser = argparse.ArgumentParser(description="Run cloud segmentation and type classification.")
    parser.add_argument("--config", default="configs/default.yaml")
    parser.add_argument("--image", required=True)
    parser.add_argument("--output", default=None)
    args = parser.parse_args()

    cfg = load_config(args.config)
    data_cfg = cfg["data"]
    inf_cfg = cfg["inference"]
    det_cfg = cfg["detector"]
    identifier = CloudIdentifier(
        detector_weights=inf_cfg["detector_weights"],
        classifier_weights=inf_cfg["classifier_weights"],
        class_names=data_cfg["classification_classes"],
        detector_backend=det_cfg.get("backend", "yolo"),
        unet_weights=cfg.get("unet", {}).get("checkpoint"),
        unet_threshold=cfg.get("unet", {}).get("threshold", 0.45),
        unet_min_area=cfg.get("unet", {}).get("min_area", 256),
        device=get_device(cfg),
        image_size=data_cfg["image_size"],
        detector_conf=det_cfg["conf"],
        detector_iou=det_cfg["iou"],
        half=det_cfg["half"],
        crop_padding=inf_cfg["crop_padding"],
    )
    overlay, predictions = identifier.predict(args.image)
    output = Path(args.output) if args.output else Path(inf_cfg["output_dir"]) / Path(args.image).name
    output.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output), overlay)
    for pred in predictions:
        print(
            f"{pred.class_name}: class={pred.class_confidence:.3f} "
            f"detector={pred.detector_confidence:.3f} box={pred.box}"
        )
    print(f"saved={output}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile export.py
from __future__ import annotations

import argparse
import importlib.util
from pathlib import Path

import torch

from cloud_chaser.config import get_device, load_config
from cloud_chaser.models.classifier import CloudClassifier
from cloud_chaser.utils.checkpoint import load_checkpoint


def export_detector(weights: str, fmt: str, imgsz: int, half: bool) -> None:
    from ultralytics import YOLO

    YOLO(weights).export(format=fmt, imgsz=imgsz, half=half)


def export_classifier(cfg: dict, fmt: str, output: str | None = None) -> Path:
    device = get_device(cfg)
    checkpoint_path = cfg["classifier"]["checkpoint"]
    checkpoint = load_checkpoint(checkpoint_path, map_location=device)
    model = CloudClassifier(
        num_classes=len(checkpoint["classes"]),
        backbone=checkpoint["backbone"],
        dropout=0.0,
        pretrained=False,
    ).to(device)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    image_size = cfg["data"]["image_size"]
    dummy = torch.randn(1, 3, image_size, image_size, device=device)
    output_path = Path(output or Path(checkpoint_path).with_suffix(f".{fmt}"))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if fmt == "torchscript":
        traced = torch.jit.trace(model, dummy)
        traced.save(str(output_path))
    elif fmt == "onnx":
        if importlib.util.find_spec("onnxscript") is None:
            print("Skipping classifier ONNX export: onnxscript is not installed.")
            return output_path
        torch.onnx.export(
            model,
            dummy,
            str(output_path),
            input_names=["image"],
            output_names=["logits"],
            dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
            opset_version=17,
        )
    else:
        raise ValueError("Classifier format must be 'torchscript' or 'onnx'")
    return output_path


def main() -> None:
    parser = argparse.ArgumentParser(description="Export detector or classifier artifacts.")
    parser.add_argument("target", choices=["detector", "classifier"])
    parser.add_argument("--config", default="configs/default.yaml")
    parser.add_argument("--format", default="onnx", choices=["onnx", "torchscript", "engine"])
    parser.add_argument("--output", default=None)
    args = parser.parse_args()

    cfg = load_config(args.config)
    if args.target == "detector":
        export_detector(
            cfg["inference"]["detector_weights"],
            args.format,
            cfg["detector"]["imgsz"],
            cfg["detector"]["half"],
        )
    else:
        output = export_classifier(cfg, args.format, args.output)
        print(f"saved={output}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile cloud_chaser/__init__.py
"""Cloud Chaser: cloud segmentation and meteorological type classification."""

__all__ = ["__version__"]
__version__ = "0.1.0"


In [ ]:
%%writefile cloud_chaser/config.py
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
from typing import Any

import yaml


def _deep_update(base: dict[str, Any], override: dict[str, Any]) -> dict[str, Any]:
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            base[key] = _deep_update(base[key], value)
        else:
            base[key] = value
    return base


def load_config(path: str | Path = "configs/default.yaml", overrides: dict[str, Any] | None = None) -> dict[str, Any]:
    """Load a YAML config and optionally apply nested dictionary overrides."""
    with Path(path).open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if overrides:
        cfg = _deep_update(deepcopy(cfg), overrides)
    return cfg


def save_yaml(data: dict[str, Any], path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, sort_keys=False)


def get_device(cfg: dict[str, Any]) -> str:
    import torch

    requested = cfg.get("project", {}).get("device", "cuda")
    if requested == "cuda" and not torch.cuda.is_available():
        return "cpu"
    return requested


In [ ]:
%%writefile cloud_chaser/data/__init__.py
"""Dataset and preprocessing utilities."""


In [ ]:
%%writefile cloud_chaser/data/ade20k.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm

from cloud_chaser.config import save_yaml


@dataclass(frozen=True)
class AdeClassMap:
    requested_names: list[str]
    class_ids: list[int]
    missing_names: list[str]


def parse_object_info(path: str | Path) -> dict[str, int]:
    mapping: dict[str, int] = {}
    with Path(path).open("r", encoding="utf-8") as f:
        next(f, None)
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 4:
                continue
            idx = int(parts[0].strip())
            names = [n.strip().lower() for n in parts[3].split(",") if n.strip()]
            for name in names:
                mapping[name] = idx
    return mapping


def resolve_ade_classes(
    root: str | Path,
    class_names: Iterable[str],
    fallback_ids: Iterable[int] | None = None,
) -> AdeClassMap:
    requested = [name.lower().strip() for name in class_names]
    info_path = Path(root) / "objectInfo150.txt"
    name_to_id = parse_object_info(info_path)
    class_ids: list[int] = []
    missing: list[str] = []
    for name in requested:
        if name in name_to_id:
            class_ids.append(name_to_id[name])
        else:
            missing.append(name)
    for idx in fallback_ids or []:
        if idx not in class_ids:
            class_ids.append(int(idx))
    if not class_ids:
        raise ValueError(
            f"None of the requested ADE classes {requested} were found in {info_path}, "
            "and no fallback IDs were supplied."
        )
    return AdeClassMap(requested_names=requested, class_ids=sorted(set(class_ids)), missing_names=missing)


def _iter_pairs(root: Path, split: str) -> list[tuple[Path, Path]]:
    image_dir = root / "images" / split
    mask_dir = root / "annotations" / split
    pairs: list[tuple[Path, Path]] = []
    for mask_path in sorted(mask_dir.glob("*.png")):
        stem = mask_path.stem
        image_path = image_dir / f"{stem}.jpg"
        if image_path.exists():
            pairs.append((image_path, mask_path))
    return pairs


def _mask_to_yolo_polygons(mask: np.ndarray, min_area: int) -> list[list[float]]:
    num_labels, labels = cv2.connectedComponents(mask.astype(np.uint8), connectivity=8)
    polygons: list[list[float]] = []
    h, w = mask.shape[:2]
    for component_id in range(1, num_labels):
        component = (labels == component_id).astype(np.uint8)
        if int(component.sum()) < min_area:
            continue
        contours, _ = cv2.findContours(component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            area = cv2.contourArea(contour)
            if area < min_area or len(contour) < 3:
                continue
            epsilon = 0.0025 * cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, epsilon, True).reshape(-1, 2)
            if len(approx) < 3:
                continue
            coords: list[float] = []
            for x, y in approx:
                coords.extend([float(np.clip(x / w, 0, 1)), float(np.clip(y / h, 0, 1))])
            if len(coords) >= 6:
                polygons.append(coords)
    return polygons


def prepare_ade20k_yolo(
    root: str | Path,
    output_dir: str | Path,
    class_names: Iterable[str],
    fallback_class_ids: Iterable[int] | None = None,
    min_mask_area: int = 512,
) -> Path:
    """Convert ADE20K semantic masks into a one-class YOLO segmentation dataset."""
    root = Path(root)
    output_dir = Path(output_dir)
    class_map = resolve_ade_classes(root, class_names, fallback_class_ids)
    split_map = {"training": "train", "validation": "val"}

    for ade_split, yolo_split in split_map.items():
        image_out = output_dir / "images" / yolo_split
        label_out = output_dir / "labels" / yolo_split
        image_out.mkdir(parents=True, exist_ok=True)
        label_out.mkdir(parents=True, exist_ok=True)

        pairs = _iter_pairs(root, ade_split)
        for image_path, mask_path in tqdm(pairs, desc=f"Preparing ADE20K {ade_split}"):
            ade_mask = np.array(Image.open(mask_path))
            binary = np.isin(ade_mask, class_map.class_ids)
            polygons = _mask_to_yolo_polygons(binary, min_area=min_mask_area)
            if not polygons:
                continue
            target_image = image_out / image_path.name
            if not target_image.exists():
                target_image.symlink_to(image_path.resolve())
            label_path = label_out / f"{image_path.stem}.txt"
            lines = ["0 " + " ".join(f"{v:.6f}" for v in polygon) for polygon in polygons]
            label_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

    data_yaml = {
        "path": str(output_dir.resolve()),
        "train": "images/train",
        "val": "images/val",
        "names": {0: "cloud"},
        "metadata": {
            "ade_class_ids": class_map.class_ids,
            "requested_ade_classes": class_map.requested_names,
            "missing_requested_classes": class_map.missing_names,
        },
    }
    save_yaml(data_yaml, output_dir / "cloud_seg.yaml")
    return output_dir / "cloud_seg.yaml"


In [ ]:
%%writefile cloud_chaser/data/augmentations.py
from __future__ import annotations

import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def random_resized_crop(image_size: int, scale: tuple[float, float], ratio: tuple[float, float]):
    """Return RandomResizedCrop for both Albumentations 1.x and 2.x APIs."""
    try:
        return A.RandomResizedCrop(size=(image_size, image_size), scale=scale, ratio=ratio)
    except Exception:
        return A.RandomResizedCrop(height=image_size, width=image_size, scale=scale, ratio=ratio)


def classification_train_transforms(
    image_size: int,
    random_shadow_p: float = 0.25,
    gaussian_blur_p: float = 0.2,
    hflip_p: float = 0.5,
    vflip_p: float = 0.15,
) -> A.Compose:
    """Meteorology-aware classification augmentation.

    Flips preserve cloud texture statistics, while blur and shadows simulate focus,
    haze, occlusion, and illumination changes seen in outdoor sky imagery.
    """
    return A.Compose(
        [
            random_resized_crop(image_size, scale=(0.65, 1.0), ratio=(0.85, 1.2)),
            A.HorizontalFlip(p=hflip_p),
            A.VerticalFlip(p=vflip_p),
            A.RandomShadow(p=random_shadow_p),
            A.GaussianBlur(blur_limit=(3, 5), p=gaussian_blur_p),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.12, hue=0.03, p=0.35),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]
    )


def eval_transforms(image_size: int) -> A.Compose:
    return A.Compose(
        [
            A.Resize(image_size, image_size),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]
    )



In [ ]:
%%writefile cloud_chaser/data/gcd.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_ALIASES = {
    "train": ("train", "training", "Train", "Training"),
    "val": ("val", "valid", "validation", "Val", "Valid", "Validation"),
    "test": ("test", "testing", "Test", "Testing"),
}
CLASS_KEYWORDS = ("cumulus", "altocumulus", "cirrus", "clearsky", "clear", "stratocumulus", "cumulonimbus", "mixed")
FORBIDDEN_CLASSIFIER_ROOTS = ("swimseg", "skyimage", "sky-image", "segmentation", "mask")


@dataclass(frozen=True)
class ImageRecord:
    path: Path
    label: int


def _is_forbidden_classifier_root(path: Path) -> bool:
    text = str(path).lower()
    return any(token in text for token in FORBIDDEN_CLASSIFIER_ROOTS)


def _has_images(path: Path) -> bool:
    return path.exists() and any(p.suffix.lower() in IMAGE_EXTENSIONS for p in path.rglob("*"))


def _image_count(path: Path) -> int:
    return sum(1 for p in path.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)


def _find_split_dir(root: Path, split: str) -> Path | None:
    for name in SPLIT_ALIASES[split]:
        candidate = root / name
        if candidate.exists() and candidate.is_dir():
            return candidate
    return None


def _class_dirs(path: Path) -> list[Path]:
    if not path.exists() or not path.is_dir():
        return []
    return sorted(p for p in path.iterdir() if p.is_dir() and _has_images(p))


def _looks_like_class_root(path: Path) -> bool:
    dirs = _class_dirs(path)
    if len(dirs) < 2:
        return False
    names = " ".join(d.name.lower().replace("_", "") for d in dirs)
    keyword_hits = sum(1 for keyword in CLASS_KEYWORDS if keyword in names)
    return keyword_hits >= 2 or len(dirs) >= 5


def _candidate_roots(root: Path) -> list[Path]:
    candidates: list[Path] = []
    if root.exists():
        candidates.append(root)
        candidates.extend(p for p in root.rglob("*") if p.is_dir())
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(p for p in kaggle_input.rglob("*") if p.is_dir())
        candidates.extend(p for p in kaggle_input.glob("*") if p.is_dir())
    seen: set[Path] = set()
    unique: list[Path] = []
    for path in candidates:
        if path not in seen:
            unique.append(path)
            seen.add(path)
    return unique


def resolve_gcd_root(root: str | Path) -> Path:
    root = Path(root)
    scored: list[tuple[int, int, Path]] = []
    for path in _candidate_roots(root):
        if _is_forbidden_classifier_root(path):
            continue
        train_dir = _find_split_dir(path, "train")
        if train_dir is not None and _class_dirs(train_dir):
            scored.append((_image_count(train_dir), 0, path))
        elif _looks_like_class_root(path):
            scored.append((_image_count(path), 1, path))
    if scored:
        return sorted(scored, key=lambda item: (-item[0], item[1], len(item[2].parts), str(item[2])))[0][2]

    available = []
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for path in sorted(kaggle_input.glob("*")):
            available.append(f"{path} images={_image_count(path)}")
    raise FileNotFoundError(
        "Could not find GCD classification images. Attach the GCD dataset or set data.gcd_root to a folder "
        "containing either train/<class>/*.jpg or <class>/*.jpg. Available inputs: " + "; ".join(available)
    )


def discover_classes(root: str | Path) -> list[str]:
    root = resolve_gcd_root(root)
    train_root = _find_split_dir(root, "train") or root
    classes = sorted(p.name for p in _class_dirs(train_root))
    if not classes:
        raise RuntimeError(f"No class folders with images found under {train_root}")
    return classes


def _records_for_split(split_dir: Path | None, classes: list[str]) -> list[ImageRecord]:
    if split_dir is None or not split_dir.exists():
        return []
    records: list[ImageRecord] = []
    class_to_idx = {name: idx for idx, name in enumerate(classes)}
    available = {p.name.lower(): p for p in split_dir.iterdir() if p.is_dir()}
    for class_name in classes:
        class_dir = split_dir / class_name
        if not class_dir.exists():
            class_dir = available.get(class_name.lower())
        if class_dir is None or not class_dir.exists():
            continue
        for path in sorted(class_dir.rglob("*")):
            if path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append(ImageRecord(path=path, label=class_to_idx[class_name]))
    return records


def _split_records(records: list[ImageRecord], split: str, val_fraction: float, seed: int) -> list[ImageRecord]:
    labels = [r.label for r in records]
    indices = list(range(len(records)))
    train_idx, holdout_idx = train_test_split(indices, test_size=val_fraction * 2, random_state=seed, stratify=labels)
    holdout_labels = [records[i].label for i in holdout_idx]
    val_idx, test_idx = train_test_split(holdout_idx, test_size=0.5, random_state=seed, stratify=holdout_labels)
    selected = {"train": train_idx, "val": val_idx, "test": test_idx}[split]
    return [records[i] for i in selected]


def build_gcd_records(
    root: str | Path,
    split: str,
    classes: list[str] | None = None,
    val_fraction: float = 0.15,
    seed: int = 42,
) -> tuple[list[ImageRecord], list[str]]:
    root = resolve_gcd_root(root)
    discovered = discover_classes(root)
    classes = classes or discovered

    train_dir = _find_split_dir(root, "train")
    probe_root = train_dir or root
    probe_records = _records_for_split(probe_root, classes)
    if not probe_records:
        print(f"Configured GCD classes {classes} did not match folders under {probe_root}.")
        print(f"Using discovered classes instead: {discovered}")
        classes = discovered

    if train_dir is None:
        all_records = _records_for_split(root, classes)
        if not all_records:
            raise RuntimeError(f"No GCD images found under class folders in {root}")
        return _split_records(all_records, split, val_fraction, seed), classes

    if split == "test":
        records = _records_for_split(_find_split_dir(root, "test"), classes)
        if records:
            return records, classes
        all_train_records = _records_for_split(train_dir, classes)
        return _split_records(all_train_records, "test", val_fraction, seed), classes

    val_dir = _find_split_dir(root, "val")
    if split == "val" and val_dir is not None:
        records = _records_for_split(val_dir, classes)
        if records:
            return records, classes

    train_records = _records_for_split(train_dir, classes)
    if not train_records:
        raise RuntimeError(f"No GCD train images found under {train_dir}. Check data.gcd_root.")
    labels = [r.label for r in train_records]
    indices = list(range(len(train_records)))
    train_idx, val_idx = train_test_split(indices, test_size=val_fraction, random_state=seed, stratify=labels)
    selected = train_idx if split == "train" else val_idx
    return [train_records[i] for i in selected], classes


class GCDDataset(Dataset):
    def __init__(
        self,
        root: str | Path,
        split: str,
        transform=None,
        classes: list[str] | None = None,
        val_fraction: float = 0.15,
        seed: int = 42,
    ) -> None:
        self.records, self.classes = build_gcd_records(root, split, classes, val_fraction, seed)
        self.transform = transform
        print(f"GCD {split}: {len(self.records)} images, classes={self.classes}")

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int):
        record = self.records[index]
        image = cv2.cvtColor(cv2.imread(str(record.path)), cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)["image"]
        else:
            image = Image.open(record.path).convert("RGB")
        return image, record.label


In [ ]:
%%writefile cloud_chaser/data/swimseg.py
from __future__ import annotations

import csv
import re
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset
from tqdm import tqdm

from cloud_chaser.config import save_yaml
from cloud_chaser.data.augmentations import IMAGENET_MEAN, IMAGENET_STD
from cloud_chaser.data.gcd import IMAGE_EXTENSIONS

MASK_TOKENS = (
    "mask",
    "masks",
    "gt",
    "gtmaps",
    "groundtruth",
    "ground_truth",
    "truth",
    "label",
    "labels",
    "annotation",
    "annotations",
    "segmentation",
    "binary",
)


@dataclass(frozen=True)
class SwimsegPair:
    image_path: Path
    mask_path: Path


def _is_probable_binary_mask(path: Path) -> bool:
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        return False
    sample = image[:: max(1, image.shape[0] // 256), :: max(1, image.shape[1] // 256)]
    unique = np.unique(sample)
    if len(unique) <= 8:
        return True
    low_high = ((sample < 16) | (sample > 239)).mean()
    return bool(low_high > 0.97)


def _looks_like_mask_path(path: Path) -> bool:
    joined = "/".join(part.lower() for part in path.parts)
    return any(token in joined for token in MASK_TOKENS) or _is_probable_binary_mask(path)


def _normal_stem(path: Path) -> str:
    stem = path.stem.lower()
    stem = re.sub(
        r"(_|-)?(mask|gt|gtmap|groundtruth|ground_truth|truth|label|annotation|segmentation|binary)$",
        "",
        stem,
    )
    return re.sub(r"[^a-z0-9]+", "", stem)


def discover_swimseg_pairs(root: str | Path) -> list[SwimsegPair]:
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"SWIMSEG root does not exist: {root}")

    files = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    if not files:
        raise RuntimeError(f"No image files found under {root}")

    mask_set = {p for p in files if _looks_like_mask_path(p)}
    mask_files = sorted(mask_set)
    image_files = sorted(p for p in files if p not in mask_set)

    if not image_files or not mask_files:
        mask_set = {p for p in files if _is_probable_binary_mask(p)}
        mask_files = sorted(mask_set)
        image_files = sorted(p for p in files if p not in mask_set)

    masks_by_key: dict[str, list[Path]] = {}
    for mask in mask_files:
        masks_by_key.setdefault(_normal_stem(mask), []).append(mask)

    pairs: list[SwimsegPair] = []
    used_masks: set[Path] = set()
    for image in image_files:
        candidates = [m for m in masks_by_key.get(_normal_stem(image), []) if m not in used_masks]
        if candidates:
            mask = candidates[0]
            pairs.append(SwimsegPair(image, mask))
            used_masks.add(mask)

    # Some packaged SWIMSEG variants keep images and masks aligned by sorted order.
    if len(pairs) < min(len(image_files), len(mask_files)) * 0.5:
        pairs = []
        for image, mask in zip(sorted(image_files), sorted(mask_files), strict=False):
            img = cv2.imread(str(image), cv2.IMREAD_COLOR)
            msk = cv2.imread(str(mask), cv2.IMREAD_GRAYSCALE)
            if img is not None and msk is not None and img.shape[:2] == msk.shape[:2]:
                pairs.append(SwimsegPair(image, mask))

    if not pairs:
        raise RuntimeError(
            f"Could not pair SWIMSEG images and masks under {root}. "
            "Inspect the dataset tree and update data.swimseg_root."
        )
    print(f"Discovered {len(pairs)} SWIMSEG image/mask pairs under {root}")
    return pairs


def _mask_to_yolo_polygons(mask: np.ndarray, min_area: int) -> list[list[float]]:
    mask = mask.astype(np.uint8)
    num_labels, labels = cv2.connectedComponents(mask, connectivity=8)
    h, w = mask.shape[:2]
    polygons: list[list[float]] = []
    for component_id in range(1, num_labels):
        component = (labels == component_id).astype(np.uint8)
        if int(component.sum()) < min_area:
            continue
        contours, _ = cv2.findContours(component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            if cv2.contourArea(contour) < min_area or len(contour) < 3:
                continue
            epsilon = 0.0025 * cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, epsilon, True).reshape(-1, 2)
            if len(approx) < 3:
                continue
            coords: list[float] = []
            for x, y in approx:
                coords.extend([float(np.clip(x / w, 0, 1)), float(np.clip(y / h, 0, 1))])
            if len(coords) >= 6:
                polygons.append(coords)
    return polygons


def _binary_cloud_mask(mask_path: Path, invert: bool = False) -> np.ndarray:
    mask = np.array(Image.open(mask_path).convert("L"))
    binary = mask > 127
    if invert:
        binary = ~binary
    return binary.astype(np.uint8)


class SwimsegMaskDataset(Dataset):
    def __init__(
        self,
        prepared_dir: str | Path,
        split: str,
        image_size: int,
    ) -> None:
        self.prepared_dir = Path(prepared_dir)
        self.split = split
        self.image_size = image_size
        manifest = self.prepared_dir / "manifest.csv"
        if not manifest.exists():
            raise FileNotFoundError(f"SWIMSEG manifest not found: {manifest}")
        self.records: list[tuple[Path, Path]] = []
        with manifest.open("r", newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                if row["split"] == split:
                    self.records.append((Path(row["image"]), Path(row["mask"])))
        if not self.records:
            raise RuntimeError(f"No SWIMSEG records found for split={split} in {manifest}")
        self.mean = np.array(IMAGENET_MEAN, dtype=np.float32)
        self.std = np.array(IMAGENET_STD, dtype=np.float32)

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        image_path, mask_path = self.records[index]
        image = cv2.cvtColor(cv2.imread(str(image_path), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if image is None or mask is None:
            raise FileNotFoundError(f"Could not read SWIMSEG pair: {image_path}, {mask_path}")
        image = cv2.resize(image, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)
        mask = cv2.resize(mask, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)
        image = image.astype(np.float32) / 255.0
        image = (image - self.mean) / self.std
        mask = (mask > 127).astype(np.float32)
        image_tensor = torch.from_numpy(image.transpose(2, 0, 1)).float()
        mask_tensor = torch.from_numpy(mask[None]).float()
        return image_tensor, mask_tensor


def _safe_link_or_copy(source: Path, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        return
    try:
        target.symlink_to(source.resolve())
    except OSError:
        image = cv2.imread(str(source), cv2.IMREAD_UNCHANGED)
        if image is None:
            raise FileNotFoundError(source)
        cv2.imwrite(str(target), image)


def prepare_swimseg_yolo(
    root: str | Path,
    output_dir: str | Path,
    val_fraction: float = 0.1,
    test_fraction: float = 0.1,
    seed: int = 42,
    min_mask_area: int = 96,
    invert_masks: bool = False,
) -> Path:
    output_dir = Path(output_dir)
    pairs = discover_swimseg_pairs(root)
    indices = list(range(len(pairs)))
    train_idx, holdout_idx = train_test_split(
        indices,
        test_size=val_fraction + test_fraction,
        random_state=seed,
    )
    relative_test = test_fraction / max(val_fraction + test_fraction, 1e-9)
    val_idx, test_idx = train_test_split(holdout_idx, test_size=relative_test, random_state=seed)
    split_indices = {"train": train_idx, "val": val_idx, "test": test_idx}

    manifest_path = output_dir / "manifest.csv"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with manifest_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["split", "image", "mask"])
        writer.writeheader()
        for split, split_idx in split_indices.items():
            for idx in tqdm(split_idx, desc=f"Preparing SWIMSEG {split}"):
                pair = pairs[idx]
                image = cv2.imread(str(pair.image_path), cv2.IMREAD_COLOR)
                if image is None:
                    continue
                h, w = image.shape[:2]
                binary = _binary_cloud_mask(pair.mask_path, invert=invert_masks)
                if binary.shape != (h, w):
                    binary = cv2.resize(binary, (w, h), interpolation=cv2.INTER_NEAREST)
                polygons = _mask_to_yolo_polygons(binary, min_area=min_mask_area)
                if not polygons:
                    continue

                target_image = output_dir / "images" / split / pair.image_path.name
                target_mask = output_dir / "masks" / split / f"{pair.image_path.stem}.png"
                label_path = output_dir / "labels" / split / f"{pair.image_path.stem}.txt"
                _safe_link_or_copy(pair.image_path, target_image)
                target_mask.parent.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(target_mask), binary * 255)
                label_path.parent.mkdir(parents=True, exist_ok=True)
                label_path.write_text(
                    "\n".join("0 " + " ".join(f"{v:.6f}" for v in poly) for poly in polygons) + "\n",
                    encoding="utf-8",
                )
                writer.writerow({"split": split, "image": str(target_image), "mask": str(target_mask)})

    data_yaml = {
        "path": str(output_dir.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "cloud"},
        "metadata": {
            "source": "SWIMSEG cloud-mask dataset",
            "binary_mask_cloud_value": "white unless swimseg_invert_masks=true",
            "pairs_discovered": len(pairs),
        },
    }
    save_yaml(data_yaml, output_dir / "cloud_seg.yaml")
    return output_dir / "cloud_seg.yaml"


In [ ]:
%%writefile cloud_chaser/evaluation.py
from __future__ import annotations

from pathlib import Path


def evaluate_detector(cfg: dict) -> dict[str, float]:
    """Evaluate YOLO mask mAP on the prepared SWIMSEG validation split.

    On Kaggle's Torch 2.10/Ultralytics stack, an additional per-image model.predict()
    mIoU loop can fail with "Inference tensors do not track version counter" after
    validation. Ultralytics val already reports mask mAP, so this evaluator avoids
    the fragile second prediction pass and keeps notebook runs resumable.
    """
    from ultralytics import YOLO

    data_cfg = cfg["data"]
    det_cfg = cfg["detector"]
    weights = cfg["inference"]["detector_weights"]
    model = YOLO(weights)
    data_yaml = Path(data_cfg["prepared_seg_dir"]) / "cloud_seg.yaml"
    map_metrics = model.val(
        data=str(data_yaml),
        task="segment",
        imgsz=det_cfg["imgsz"],
        conf=det_cfg["conf"],
        iou=det_cfg["iou"],
        half=det_cfg["half"],
        verbose=False,
    )
    metrics = {
        "mask_map50_95": float(getattr(map_metrics.seg, "map", 0.0)),
        "mask_map50": float(getattr(map_metrics.seg, "map50", 0.0)),
    }
    print(metrics)
    return metrics


In [ ]:
%%writefile cloud_chaser/inference_pipeline.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F

from cloud_chaser.data.augmentations import IMAGENET_MEAN, IMAGENET_STD
from cloud_chaser.data.augmentations import eval_transforms
from cloud_chaser.models.classifier import CloudClassifier
from cloud_chaser.utils.checkpoint import load_checkpoint
from cloud_chaser.utils.visualization import overlay_instance


def display_class_name(class_name: str) -> str:
    name = class_name.split("_", 1)[1] if "_" in class_name and class_name[0].isdigit() else class_name
    return name.replace("_", " ").title()


@dataclass
class CloudPrediction:
    box: tuple[int, int, int, int]
    detector_confidence: float
    class_name: str
    class_confidence: float


class CloudIdentifier:
    def __init__(
        self,
        detector_weights: str | Path,
        classifier_weights: str | Path,
        class_names: list[str] | None = None,
        detector_backend: str = "yolo",
        unet_weights: str | Path | None = None,
        unet_threshold: float = 0.45,
        unet_min_area: int = 256,
        device: str = "cuda",
        image_size: int = 224,
        detector_conf: float = 0.25,
        detector_iou: float = 0.6,
        half: bool = True,
        crop_padding: int = 12,
    ) -> None:
        self.device = device if device != "cuda" or torch.cuda.is_available() else "cpu"
        if detector_backend not in {"yolo", "hybrid", "unet"}:
            raise ValueError(f"Unsupported detector backend: {detector_backend}")
        self.detector = None
        if detector_backend != "unet":
            from ultralytics import YOLO

            self.detector = YOLO(str(detector_weights))
        self.detector_conf = detector_conf
        self.detector_iou = detector_iou
        self.half = half and self.device != "cpu"
        self.crop_padding = crop_padding
        self.image_size = image_size
        self.detector_backend = detector_backend
        self.unet_threshold = unet_threshold
        self.unet_min_area = unet_min_area
        self.unet = None
        if detector_backend in {"hybrid", "unet"} and unet_weights is not None and Path(unet_weights).exists():
            from cloud_chaser.models.unet import CloudUNet

            checkpoint = load_checkpoint(unet_weights, map_location=self.device)
            self.unet = CloudUNet().to(self.device)
            self.unet.load_state_dict(checkpoint["model"])
            self.unet.eval()
        self.transform = eval_transforms(image_size)

        classifier_path = Path(classifier_weights)
        if classifier_path.suffix in {".torchscript", ".ts"}:
            if class_names is None:
                raise ValueError("class_names are required when loading a TorchScript classifier.")
            self.classes = class_names
            self.classifier = torch.jit.load(str(classifier_path), map_location=self.device).to(self.device)
        else:
            checkpoint = load_checkpoint(classifier_path, map_location=self.device)
            self.classes = checkpoint["classes"]
            self.classifier = CloudClassifier(
                num_classes=len(self.classes),
                backbone=checkpoint["backbone"],
                dropout=0.0,
                pretrained=False,
            ).to(self.device)
            self.classifier.load_state_dict(checkpoint["model"])
        self.classifier.eval()

    def _crop_instances(
        self,
        image_rgb: np.ndarray,
        masks: torch.Tensor,
        boxes: torch.Tensor,
    ) -> list[torch.Tensor]:
        h, w = image_rgb.shape[:2]
        crops: list[torch.Tensor] = []
        for _, box_tensor in zip(masks, boxes, strict=False):
            x1, y1, x2, y2 = [int(v) for v in box_tensor.tolist()]
            x1 = max(0, x1 - self.crop_padding)
            y1 = max(0, y1 - self.crop_padding)
            x2 = min(w, x2 + self.crop_padding)
            y2 = min(h, y2 + self.crop_padding)
            crop = image_rgb[y1:y2, x1:x2]
            crops.append(self.transform(image=crop)["image"])
        return crops

    def _unet_instances(self, image_rgb: np.ndarray) -> tuple[torch.Tensor, torch.Tensor, list[float]]:
        if self.unet is None:
            return torch.empty((0, *image_rgb.shape[:2]), dtype=torch.bool), torch.empty((0, 4)), []
        h, w = image_rgb.shape[:2]
        resized = cv2.resize(image_rgb, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)
        x = resized.astype(np.float32) / 255.0
        x = (x - np.array(IMAGENET_MEAN, dtype=np.float32)) / np.array(IMAGENET_STD, dtype=np.float32)
        tensor = torch.from_numpy(x.transpose(2, 0, 1))[None].float().to(self.device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=self.half):
            prob = torch.sigmoid(self.unet(tensor))[0, 0].detach().float().cpu().numpy()
        prob = cv2.resize(prob, (w, h), interpolation=cv2.INTER_LINEAR)
        binary = prob >= self.unet_threshold
        num_labels, labels = cv2.connectedComponents(binary.astype(np.uint8), connectivity=8)
        masks: list[np.ndarray] = []
        boxes: list[list[float]] = []
        scores: list[float] = []
        for component_id in range(1, num_labels):
            mask = labels == component_id
            area = int(mask.sum())
            if area < self.unet_min_area:
                continue
            ys, xs = np.where(mask)
            x1, x2 = int(xs.min()), int(xs.max()) + 1
            y1, y2 = int(ys.min()), int(ys.max()) + 1
            masks.append(mask)
            boxes.append([x1, y1, x2, y2])
            scores.append(float(prob[mask].mean()))
        if not masks:
            return torch.empty((0, h, w), dtype=torch.bool), torch.empty((0, 4)), []
        return torch.from_numpy(np.stack(masks)).bool(), torch.tensor(boxes, dtype=torch.float32), scores

    @torch.no_grad()
    def predict(self, image_path: str | Path) -> tuple[np.ndarray, list[CloudPrediction]]:
        image_bgr = cv2.imread(str(image_path))
        if image_bgr is None:
            raise FileNotFoundError(f"Could not read image: {image_path}")
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        h, w = image_rgb.shape[:2]

        if self.detector_backend == "unet":
            masks, boxes, detector_scores = self._unet_instances(image_rgb)
        else:
            if self.detector is None:
                raise RuntimeError("YOLO detector is not initialized.")
            results = self.detector.predict(
                image_rgb,
                conf=self.detector_conf,
                iou=self.detector_iou,
                retina_masks=True,
                device=self.device,
                half=self.half,
                verbose=False,
            )
            result = results[0]
            if result.masks is None or result.boxes is None or len(result.boxes) == 0:
                if self.detector_backend == "hybrid":
                    masks, boxes, detector_scores = self._unet_instances(image_rgb)
                else:
                    return image_bgr, []
            else:
                masks = result.masks.data
                if masks.shape[-2:] != (h, w):
                    masks = F.interpolate(masks[:, None].float(), size=(h, w), mode="nearest")[:, 0] > 0.5
                boxes = result.boxes.xyxy.detach().cpu()
                detector_scores = result.boxes.conf.detach().cpu().tolist()

        if len(boxes) == 0:
            return image_bgr, []

        crops = self._crop_instances(image_rgb, masks, boxes)
        batch = torch.stack(crops).to(self.device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=self.half):
            probs = torch.softmax(self.classifier(batch), dim=1)
        confs, labels = probs.max(dim=1)

        overlay = image_bgr.copy()
        predictions: list[CloudPrediction] = []
        for i, (box_tensor, det_score, label_idx, cls_score) in enumerate(
            zip(boxes, detector_scores, labels, confs, strict=False)
        ):
            box = tuple(int(v) for v in box_tensor.tolist())
            class_name = display_class_name(self.classes[int(label_idx)])
            class_conf = float(cls_score.detach().cpu())
            mask = masks[i].detach().cpu().numpy().astype(bool)
            overlay = overlay_instance(
                overlay,
                mask,
                box,
                class_name,
                class_conf,
                color_index=int(label_idx),
            )
            predictions.append(
                CloudPrediction(
                    box=box,
                    detector_confidence=float(det_score),
                    class_name=class_name,
                    class_confidence=class_conf,
                )
            )
        return overlay, predictions


In [ ]:
%%writefile cloud_chaser/models/__init__.py
"""Model definitions and wrappers."""


In [ ]:
%%writefile cloud_chaser/models/classifier.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Literal

import torch
from torch import nn
from torchvision import models

BackboneName = Literal["resnet50", "efficientnet_b0", "densenet121"]


@dataclass(frozen=True)
class BackboneBundle:
    encoder: nn.Module
    features_dim: int


class DenseNetEncoder(nn.Module):
    def __init__(self, model: models.DenseNet) -> None:
        super().__init__()
        self.features = model.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.features(x))
        x = self.pool(x)
        return torch.flatten(x, 1)


def build_encoder(backbone: BackboneName = "resnet50", pretrained: bool = True) -> BackboneBundle:
    if backbone == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        model = models.resnet50(weights=weights)
        features_dim = model.fc.in_features
        model.fc = nn.Identity()
        return BackboneBundle(model, features_dim)
    if backbone == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        features_dim = model.classifier[1].in_features
        model.classifier = nn.Identity()
        return BackboneBundle(model, features_dim)
    if backbone == "densenet121":
        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.densenet121(weights=weights)
        return BackboneBundle(DenseNetEncoder(model), model.classifier.in_features)
    raise ValueError(f"Unsupported backbone: {backbone}")


class CloudClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        backbone: BackboneName = "resnet50",
        dropout: float = 0.2,
        pretrained: bool = True,
    ) -> None:
        super().__init__()
        bundle = build_encoder(backbone, pretrained=pretrained)
        self.backbone_name = backbone
        self.encoder = bundle.encoder
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(bundle.features_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.encoder(x))

    def freeze_encoder(self, freeze: bool = True) -> None:
        for param in self.encoder.parameters():
            param.requires_grad = not freeze


In [ ]:
%%writefile cloud_chaser/models/detector.py
from __future__ import annotations

from pathlib import Path


def train_yolo_segmenter(
    model_name_or_path: str,
    data_yaml: str | Path,
    output_dir: str | Path,
    epochs: int,
    imgsz: int,
    batch: int,
    device: str,
    patience: int = 20,
    lr0: float = 0.01,
    weight_decay: float = 0.0005,
):
    from ultralytics import YOLO

    output_dir = Path(output_dir)
    last_checkpoint = output_dir / "weights" / "last.pt"
    best_checkpoint = output_dir / "weights" / "best.pt"
    resume = last_checkpoint.exists()
    weights = str(last_checkpoint) if resume else str(best_checkpoint) if best_checkpoint.exists() else model_name_or_path
    yolo = YOLO(weights)
    return yolo.train(
        data=str(data_yaml),
        task="segment",
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        device=device,
        project=str(output_dir.parent),
        name=output_dir.name,
        patience=patience,
        lr0=lr0,
        weight_decay=weight_decay,
        amp=True,
        resume=resume,
        exist_ok=True,
    )


In [ ]:
%%writefile cloud_chaser/models/unet.py
from __future__ import annotations

import torch
from torch import nn


class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_channels, out_channels))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        diff_y = skip.size(2) - x.size(2)
        diff_x = skip.size(3) - x.size(3)
        x = nn.functional.pad(
            x,
            [diff_x // 2, diff_x - diff_x // 2, diff_y // 2, diff_y - diff_y // 2],
        )
        return self.conv(torch.cat([skip, x], dim=1))


class CloudUNet(nn.Module):
    """Compact U-Net for binary cloud semantic segmentation."""

    def __init__(self, base_channels: int = 32) -> None:
        super().__init__()
        self.inc = DoubleConv(3, base_channels)
        self.down1 = Down(base_channels, base_channels * 2)
        self.down2 = Down(base_channels * 2, base_channels * 4)
        self.down3 = Down(base_channels * 4, base_channels * 8)
        self.down4 = Down(base_channels * 8, base_channels * 16)
        self.up1 = Up(base_channels * 16, base_channels * 8)
        self.up2 = Up(base_channels * 8, base_channels * 4)
        self.up3 = Up(base_channels * 4, base_channels * 2)
        self.up4 = Up(base_channels * 2, base_channels)
        self.out = nn.Conv2d(base_channels, 1, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.out(x)


In [ ]:
%%writefile cloud_chaser/training.py
from __future__ import annotations

from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from cloud_chaser.config import get_device
from cloud_chaser.data.augmentations import (
    classification_train_transforms,
    eval_transforms,
)
from cloud_chaser.data.gcd import GCDDataset
from cloud_chaser.data.swimseg import SwimsegMaskDataset, prepare_swimseg_yolo
from cloud_chaser.models.classifier import CloudClassifier
from cloud_chaser.models.detector import train_yolo_segmenter
from cloud_chaser.models.unet import CloudUNet
from cloud_chaser.utils.checkpoint import load_checkpoint, save_checkpoint
from cloud_chaser.utils.metrics import classification_metrics
from cloud_chaser.utils.seed import seed_everything


def train_detector(cfg: dict) -> None:
    seed_everything(cfg["project"]["seed"])
    data_cfg = cfg["data"]
    detector_cfg = cfg["detector"]
    data_yaml = prepare_swimseg_yolo(
        root=data_cfg["swimseg_root"],
        output_dir=data_cfg["prepared_seg_dir"],
        val_fraction=data_cfg.get("seg_val_fraction", 0.1),
        test_fraction=data_cfg.get("seg_test_fraction", 0.1),
        seed=cfg["project"]["seed"],
        min_mask_area=data_cfg.get("min_mask_area", 96),
        invert_masks=data_cfg.get("swimseg_invert_masks", False),
    )
    output_dir = Path(cfg["project"]["output_dir"]) / "detector"
    train_yolo_segmenter(
        model_name_or_path=detector_cfg["model"],
        data_yaml=data_yaml,
        output_dir=output_dir,
        epochs=detector_cfg["epochs"],
        imgsz=detector_cfg["imgsz"],
        batch=detector_cfg["batch"],
        device=get_device(cfg),
        patience=detector_cfg["patience"],
        lr0=detector_cfg["lr0"],
        weight_decay=detector_cfg["weight_decay"],
    )


def _segmentation_scores(logits: torch.Tensor, masks: torch.Tensor) -> tuple[float, float]:
    preds = torch.sigmoid(logits) > 0.5
    targets = masks > 0.5
    intersection = (preds & targets).sum(dim=(1, 2, 3)).float()
    union = (preds | targets).sum(dim=(1, 2, 3)).float().clamp_min(1.0)
    pred_sum = preds.sum(dim=(1, 2, 3)).float()
    target_sum = targets.sum(dim=(1, 2, 3)).float()
    iou = (intersection / union).mean().item()
    dice = ((2 * intersection + 1.0) / (pred_sum + target_sum + 1.0)).mean().item()
    return iou, dice


def _run_unet_epoch(
    model: CloudUNet,
    loader: DataLoader,
    criterion: nn.Module,
    device: str,
    optimizer: torch.optim.Optimizer | None = None,
) -> dict[str, float]:
    training = optimizer is not None
    model.train(training)
    scaler = torch.cuda.amp.GradScaler(enabled=training and device == "cuda")
    total_loss = 0.0
    total_iou = 0.0
    total_dice = 0.0
    for images, masks in tqdm(loader, desc="unet-train" if training else "unet-eval"):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        with torch.set_grad_enabled(training):
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
                logits = model(images)
                bce = criterion(logits, masks)
                probs = torch.sigmoid(logits)
                intersection = (probs * masks).sum(dim=(1, 2, 3))
                dice_loss = 1 - ((2 * intersection + 1) / (probs.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3)) + 1)).mean()
                loss = bce + dice_loss
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        iou, dice = _segmentation_scores(logits.detach(), masks.detach())
        total_loss += float(loss.detach().cpu())
        total_iou += iou
        total_dice += dice
    count = max(1, len(loader))
    return {"loss": total_loss / count, "miou": total_iou / count, "dice": total_dice / count}


def train_unet_detector(cfg: dict) -> None:
    seed_everything(cfg["project"]["seed"])
    device = get_device(cfg)
    data_cfg = cfg["data"]
    unet_cfg = cfg["unet"]
    prepare_swimseg_yolo(
        root=data_cfg["swimseg_root"],
        output_dir=data_cfg["prepared_seg_dir"],
        val_fraction=data_cfg.get("seg_val_fraction", 0.1),
        test_fraction=data_cfg.get("seg_test_fraction", 0.1),
        seed=cfg["project"]["seed"],
        min_mask_area=data_cfg.get("min_mask_area", 96),
        invert_masks=data_cfg.get("swimseg_invert_masks", False),
    )
    train_ds = SwimsegMaskDataset(data_cfg["prepared_seg_dir"], "train", data_cfg["image_size"])
    val_ds = SwimsegMaskDataset(data_cfg["prepared_seg_dir"], "val", data_cfg["image_size"])
    train_loader = DataLoader(
        train_ds,
        batch_size=unet_cfg["batch_size"],
        shuffle=True,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=unet_cfg["batch_size"],
        shuffle=False,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
    )
    model = CloudUNet().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=unet_cfg["lr"], weight_decay=unet_cfg["weight_decay"])
    output_dir = Path(cfg["project"]["output_dir"]) / "unet"
    last_checkpoint = output_dir / "last.pt"
    best_checkpoint = output_dir / "best.pt"
    best_miou = -1.0
    start_epoch = 0
    resume_checkpoint = last_checkpoint if last_checkpoint.exists() else best_checkpoint if best_checkpoint.exists() else None
    if resume_checkpoint is not None:
        checkpoint = load_checkpoint(resume_checkpoint, map_location=device)
        model.load_state_dict(checkpoint["model"])
        if "optimizer" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer"])
        start_epoch = int(checkpoint.get("epoch", -1)) + 1
        best_miou = float(checkpoint.get("best_miou", checkpoint.get("val_metrics", {}).get("miou", -1.0)))
        print(f"Resuming U-Net from {resume_checkpoint} at epoch {start_epoch + 1}")
    for epoch in range(start_epoch, unet_cfg["epochs"]):
        train_metrics = _run_unet_epoch(model, train_loader, criterion, device, optimizer)
        val_metrics = _run_unet_epoch(model, val_loader, criterion, device)
        print(
            f"unet_epoch={epoch + 1} train_loss={train_metrics['loss']:.4f} "
            f"val_miou={val_metrics['miou']:.4f} val_dice={val_metrics['dice']:.4f}"
        )
        payload = {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "val_metrics": val_metrics,
            "best_miou": max(best_miou, val_metrics["miou"]),
        }
        save_checkpoint(payload, output_dir / "last.pt")
        if val_metrics["miou"] > best_miou:
            best_miou = val_metrics["miou"]
            save_checkpoint(payload, output_dir / "best.pt")


def evaluate_unet_detector(cfg: dict) -> dict[str, float]:
    device = get_device(cfg)
    data_cfg = cfg["data"]
    checkpoint = load_checkpoint(cfg["unet"]["checkpoint"], map_location=device)
    dataset = SwimsegMaskDataset(data_cfg["prepared_seg_dir"], "test", data_cfg["image_size"])
    loader = DataLoader(
        dataset,
        batch_size=cfg["unet"]["batch_size"],
        shuffle=False,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
    )
    model = CloudUNet().to(device)
    model.load_state_dict(checkpoint["model"])
    metrics = _run_unet_epoch(model, loader, nn.BCEWithLogitsLoss(), device)
    print(metrics)
    return metrics


def _run_classifier_epoch(
    model: CloudClassifier,
    loader: DataLoader,
    criterion: nn.Module,
    device: str,
    optimizer: torch.optim.Optimizer | None = None,
    amp: bool = True,
) -> dict[str, float]:
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    all_logits: list[torch.Tensor] = []
    all_targets: list[torch.Tensor] = []
    scaler = torch.cuda.amp.GradScaler(enabled=training and amp and device == "cuda")
    for images, targets in tqdm(loader, desc="train" if training else "eval"):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with torch.set_grad_enabled(training):
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp and device == "cuda"):
                logits = model(images)
                loss = criterion(logits, targets)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        total_loss += float(loss.detach().cpu())
        all_logits.append(logits.detach().cpu())
        all_targets.append(targets.detach().cpu())
    logits = torch.cat(all_logits)
    targets = torch.cat(all_targets)
    metrics = classification_metrics(logits, targets)
    metrics["loss"] = total_loss / max(1, len(loader))
    return metrics


def train_classifier(cfg: dict) -> None:
    seed_everything(cfg["project"]["seed"])
    device = get_device(cfg)
    data_cfg = cfg["data"]
    cls_cfg = cfg["classifier"]
    train_tfms = classification_train_transforms(data_cfg["image_size"], **cfg["augmentation"])
    eval_tfms = eval_transforms(data_cfg["image_size"])
    train_ds = GCDDataset(
        data_cfg["gcd_root"],
        "train",
        train_tfms,
        classes=data_cfg["classification_classes"],
        val_fraction=data_cfg["val_fraction"],
        seed=cfg["project"]["seed"],
    )
    val_ds = GCDDataset(
        data_cfg["gcd_root"],
        "val",
        eval_tfms,
        classes=train_ds.classes,
        val_fraction=data_cfg["val_fraction"],
        seed=cfg["project"]["seed"],
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=cls_cfg["batch_size"],
        shuffle=True,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cls_cfg["batch_size"],
        shuffle=False,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
    )
    model = CloudClassifier(
        num_classes=len(train_ds.classes),
        backbone=cls_cfg["backbone"],
        dropout=cls_cfg["dropout"],
    ).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cls_cfg["lr"], weight_decay=cls_cfg["weight_decay"])
    best_f1 = -1.0
    output_dir = Path(cfg["project"]["output_dir"]) / "classifier"
    last_checkpoint = output_dir / "last.pt"
    best_checkpoint = output_dir / "best.pt"
    start_epoch = 0
    resume_checkpoint = last_checkpoint if last_checkpoint.exists() else best_checkpoint if best_checkpoint.exists() else None
    if resume_checkpoint is not None:
        checkpoint = load_checkpoint(resume_checkpoint, map_location=device)
        model.load_state_dict(checkpoint["model"])
        if "optimizer" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer"])
        start_epoch = int(checkpoint.get("epoch", -1)) + 1
        best_f1 = float(checkpoint.get("best_f1", checkpoint.get("val_metrics", {}).get("f1_macro", -1.0)))
        print(f"Resuming classifier from {resume_checkpoint} at epoch {start_epoch + 1}")

    for epoch in range(start_epoch, cls_cfg["epochs"]):
        model.freeze_encoder(epoch < cls_cfg.get("freeze_backbone_epochs", 0))
        train_metrics = _run_classifier_epoch(
            model, train_loader, criterion, device, optimizer=optimizer, amp=cls_cfg["amp"]
        )
        val_metrics = _run_classifier_epoch(model, val_loader, criterion, device, amp=cls_cfg["amp"])
        print(
            f"epoch={epoch + 1} train_loss={train_metrics['loss']:.4f} "
            f"val_top1={val_metrics['top1']:.4f} val_f1={val_metrics['f1_macro']:.4f}"
        )
        payload = {
            "epoch": epoch,
            "classes": train_ds.classes,
            "backbone": cls_cfg["backbone"],
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "val_metrics": val_metrics,
            "best_f1": max(best_f1, val_metrics["f1_macro"]),
        }
        save_checkpoint(payload, output_dir / "last.pt")
        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            save_checkpoint(payload, output_dir / "best.pt")


def evaluate_classifier(cfg: dict) -> dict[str, float]:
    device = get_device(cfg)
    data_cfg = cfg["data"]
    cls_cfg = cfg["classifier"]
    checkpoint = load_checkpoint(cls_cfg["checkpoint"], map_location=device)
    dataset = GCDDataset(
        data_cfg["gcd_root"],
        "test",
        eval_transforms(data_cfg["image_size"]),
        classes=checkpoint["classes"],
    )
    loader = DataLoader(
        dataset,
        batch_size=cls_cfg["batch_size"],
        shuffle=False,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
    )
    model = CloudClassifier(
        num_classes=len(checkpoint["classes"]),
        backbone=checkpoint["backbone"],
        dropout=0.0,
        pretrained=False,
    ).to(device)
    model.load_state_dict(checkpoint["model"])
    metrics = _run_classifier_epoch(model, loader, nn.CrossEntropyLoss(), device, amp=cls_cfg["amp"])
    print(metrics)
    return metrics


In [ ]:
%%writefile cloud_chaser/utils/__init__.py
"""Shared utilities."""


In [ ]:
%%writefile cloud_chaser/utils/checkpoint.py
from __future__ import annotations

from pathlib import Path
from typing import Any

import torch


def save_checkpoint(payload: dict[str, Any], path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)


def load_checkpoint(path: str | Path, map_location: str | torch.device = "cpu") -> dict[str, Any]:
    return torch.load(Path(path), map_location=map_location)


In [ ]:
%%writefile cloud_chaser/utils/metrics.py
from __future__ import annotations

import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score


def classification_metrics(logits: torch.Tensor, targets: torch.Tensor) -> dict[str, float]:
    preds = logits.argmax(dim=1).detach().cpu().numpy()
    labels = targets.detach().cpu().numpy()
    return {
        "top1": float(accuracy_score(labels, preds)),
        "f1_macro": float(f1_score(labels, preds, average="macro", zero_division=0)),
    }


def binary_iou(pred: np.ndarray, target: np.ndarray) -> float:
    pred = pred.astype(bool)
    target = target.astype(bool)
    union = np.logical_or(pred, target).sum()
    if union == 0:
        return 1.0
    return float(np.logical_and(pred, target).sum() / union)


In [ ]:
%%writefile cloud_chaser/utils/seed.py
from __future__ import annotations

import os
import random

import numpy as np
import torch


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True


In [ ]:
%%writefile cloud_chaser/utils/visualization.py
from __future__ import annotations

import cv2
import numpy as np


PALETTE = [
    (42, 157, 143),
    (233, 196, 106),
    (230, 111, 81),
    (38, 70, 83),
    (131, 197, 190),
    (239, 71, 111),
    (17, 138, 178),
]


def overlay_instance(
    image: np.ndarray,
    mask: np.ndarray,
    box: tuple[int, int, int, int],
    label: str,
    score: float,
    color_index: int,
    alpha: float = 0.42,
) -> np.ndarray:
    color = PALETTE[color_index % len(PALETTE)]
    out = image.copy()
    color_layer = np.zeros_like(out)
    color_layer[:, :] = color
    out[mask] = cv2.addWeighted(out, 1 - alpha, color_layer, alpha, 0)[mask]
    x1, y1, x2, y2 = box
    cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
    text = f"{label} - {score:.0%}"
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    y_text = max(0, y1 - th - 8)
    cv2.rectangle(out, (x1, y_text), (x1 + tw + 8, y_text + th + 8), color, -1)
    cv2.putText(
        out,
        text,
        (x1 + 4, y_text + th + 4),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2,
        cv2.LINE_AA,
    )
    return out


In [ ]:
%%writefile scripts/__init__.py
"""Command-line helper scripts."""


In [ ]:
%%writefile scripts/build_cloud_crops.py
from __future__ import annotations

import argparse
from pathlib import Path

import cv2
from tqdm import tqdm
from ultralytics import YOLO

from cloud_chaser.data.gcd import IMAGE_EXTENSIONS


def crop_dataset(input_root: Path, output_root: Path, detector_weights: str, conf: float, padding: int) -> None:
    model = YOLO(detector_weights)
    paths = sorted(p for p in input_root.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)
    for path in tqdm(paths, desc="Cropping cloud regions"):
        image_bgr = cv2.imread(str(path))
        if image_bgr is None:
            continue
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        result = model.predict(image_rgb, conf=conf, retina_masks=True, verbose=False)[0]
        if result.boxes is None or len(result.boxes) == 0:
            rel = path.relative_to(input_root)
            target = output_root / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(target), image_bgr)
            continue
        h, w = image_bgr.shape[:2]
        boxes = result.boxes.xyxy.detach().cpu().numpy()
        for idx, box in enumerate(boxes):
            x1, y1, x2, y2 = [int(v) for v in box]
            x1 = max(0, x1 - padding)
            y1 = max(0, y1 - padding)
            x2 = min(w, x2 + padding)
            y2 = min(h, y2 + padding)
            crop = image_bgr[y1:y2, x1:x2]
            rel = path.relative_to(input_root)
            target = output_root / rel.parent / f"{rel.stem}_cloud{idx}{rel.suffix}"
            target.parent.mkdir(parents=True, exist_ok=True)
            if crop.size > 0:
                cv2.imwrite(str(target), crop)


def main() -> None:
    parser = argparse.ArgumentParser(description="Build classifier crop folders from detector boxes.")
    parser.add_argument("--input-root", required=True, type=Path)
    parser.add_argument("--output-root", required=True, type=Path)
    parser.add_argument("--detector-weights", required=True)
    parser.add_argument("--conf", type=float, default=0.25)
    parser.add_argument("--padding", type=int, default=12)
    args = parser.parse_args()
    crop_dataset(args.input_root, args.output_root, args.detector_weights, args.conf, args.padding)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/prepare_ade20k_yolo.py
from __future__ import annotations

import argparse

from cloud_chaser.config import load_config
from cloud_chaser.data.ade20k import prepare_ade20k_yolo


def main() -> None:
    parser = argparse.ArgumentParser(description="Convert ADE20K cloud/sky masks to YOLO segmentation format.")
    parser.add_argument("--config", default="configs/default.yaml")
    args = parser.parse_args()
    cfg = load_config(args.config)
    data_cfg = cfg["data"]
    path = prepare_ade20k_yolo(
        root=data_cfg["ade20k_root"],
        output_dir=data_cfg["prepared_seg_dir"],
        class_names=data_cfg["ade_classes"],
        fallback_class_ids=data_cfg.get("ade_fallback_class_ids", []),
        min_mask_area=data_cfg["min_mask_area"],
    )
    print(f"wrote={path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/prepare_swimseg_yolo.py
from __future__ import annotations

import argparse

from cloud_chaser.config import load_config
from cloud_chaser.data.swimseg import prepare_swimseg_yolo


def main() -> None:
    parser = argparse.ArgumentParser(description="Convert SWIMSEG cloud masks to YOLO segmentation format.")
    parser.add_argument("--config", default="configs/default.yaml")
    args = parser.parse_args()
    cfg = load_config(args.config)
    data_cfg = cfg["data"]
    path = prepare_swimseg_yolo(
        root=data_cfg["swimseg_root"],
        output_dir=data_cfg["prepared_seg_dir"],
        val_fraction=data_cfg.get("seg_val_fraction", 0.1),
        test_fraction=data_cfg.get("seg_test_fraction", 0.1),
        seed=cfg["project"]["seed"],
        min_mask_area=data_cfg.get("min_mask_area", 96),
        invert_masks=data_cfg.get("swimseg_invert_masks", False),
    )
    print(f"wrote={path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/gcd_visual_report.py
from __future__ import annotations

import argparse
import json
import math
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from cloud_chaser.config import get_device, load_config
from cloud_chaser.data.gcd import build_gcd_records
from cloud_chaser.inference_pipeline import CloudIdentifier, display_class_name
from cloud_chaser.utils.seed import seed_everything


def _is_clear_sky(class_name: str) -> bool:
    normalized = class_name.lower().replace("_", "").replace("-", "")
    return "clearsky" in normalized or normalized.endswith("clear") or "clear" in normalized


def _best_image_prediction(predictions) -> tuple[str | None, float]:
    if not predictions:
        return None, 0.0
    best = max(predictions, key=lambda p: p.detector_confidence * p.class_confidence)
    return best.class_name, best.detector_confidence * best.class_confidence


def _report_path(output_dir: Path, prefix: str, suffix: str) -> Path:
    return output_dir / f"{prefix}_{suffix}"


def _evaluate_cascade(
    cfg: dict,
    output_dir: Path,
    samples: int,
    backend: str,
    prefix: str,
) -> tuple[dict, list[dict]]:
    records, classes = build_gcd_records(
        cfg["data"]["gcd_root"],
        split="val",
        classes=cfg["data"].get("classification_classes"),
        val_fraction=cfg["data"]["val_fraction"],
        seed=cfg["project"]["seed"],
    )
    clear_sky = [_is_clear_sky(name) for name in classes]
    identifier = CloudIdentifier(
        detector_weights=cfg["inference"]["detector_weights"],
        classifier_weights=cfg["inference"]["classifier_weights"],
        class_names=classes,
        detector_backend=backend,
        unet_weights=cfg.get("unet", {}).get("checkpoint"),
        unet_threshold=cfg.get("unet", {}).get("threshold", 0.45),
        unet_min_area=cfg.get("unet", {}).get("min_area", 256),
        device=get_device(cfg),
        image_size=cfg["data"]["image_size"],
        detector_conf=cfg["detector"]["conf"],
        detector_iou=cfg["detector"]["iou"],
        half=cfg["detector"]["half"],
        crop_padding=cfg["inference"]["crop_padding"],
    )

    n = len(classes)
    total_by_class = np.zeros(n, dtype=np.int64)
    detection_correct_by_class = np.zeros(n, dtype=np.int64)
    detected_by_class = np.zeros(n, dtype=np.int64)
    classified_correct_by_class = np.zeros(n, dtype=np.int64)
    classified_total_by_class = np.zeros(n, dtype=np.int64)
    cascade_correct_by_class = np.zeros(n, dtype=np.int64)
    confusion = np.zeros((n, n), dtype=np.int64)

    details: list[dict] = []
    display_to_idx = {display_class_name(name): idx for idx, name in enumerate(classes)}

    for record in tqdm(records, desc="GCD cascade validation"):
        true_idx = int(record.label)
        true_name = display_class_name(classes[true_idx])
        expects_cloud = not clear_sky[true_idx]
        _, predictions = identifier.predict(record.path)
        has_detection = len(predictions) > 0
        pred_name, pred_score = _best_image_prediction(predictions)
        pred_idx = display_to_idx.get(pred_name) if pred_name is not None else None

        detection_correct = has_detection if expects_cloud else not has_detection
        classification_correct = pred_idx == true_idx if has_detection and pred_idx is not None else False
        if expects_cloud:
            cascade_correct = has_detection and classification_correct
        else:
            cascade_correct = not has_detection

        total_by_class[true_idx] += 1
        detected_by_class[true_idx] += int(has_detection)
        detection_correct_by_class[true_idx] += int(detection_correct)
        cascade_correct_by_class[true_idx] += int(cascade_correct)
        if has_detection and pred_idx is not None:
            classified_total_by_class[true_idx] += 1
            classified_correct_by_class[true_idx] += int(classification_correct)
            confusion[true_idx, pred_idx] += 1

        details.append(
            {
                "path": str(record.path),
                "true_class": true_name,
                "expects_cloud": expects_cloud,
                "has_detection": has_detection,
                "detection_correct": bool(detection_correct),
                "predicted_class": pred_name,
                "prediction_score": float(pred_score),
                "classification_correct": bool(classification_correct),
                "cascade_correct": bool(cascade_correct),
                "num_detections": len(predictions),
            }
        )

    detection_accuracy = float(detection_correct_by_class.sum() / max(total_by_class.sum(), 1))
    conditional_classification_accuracy = float(
        classified_correct_by_class.sum() / max(classified_total_by_class.sum(), 1)
    )
    cascade_accuracy = float(cascade_correct_by_class.sum() / max(total_by_class.sum(), 1))

    metrics = {
        "split": "val",
        "detector_backend": backend,
        "num_images": int(total_by_class.sum()),
        "classes": classes,
        "note": (
            "GCD has image-level class labels but no cloud masks. Detection is evaluated as an "
            "image-level cascade gate: non-clearsky classes should produce at least one cloud "
            "detection, while clearsky should produce none."
        ),
        "detector_image_accuracy": detection_accuracy,
        "classifier_accuracy_given_detection": conditional_classification_accuracy,
        "cascade_accuracy": cascade_accuracy,
        "total_by_class": total_by_class.tolist(),
        "detected_by_class": detected_by_class.tolist(),
        "detection_correct_by_class": detection_correct_by_class.tolist(),
        "classified_total_by_class": classified_total_by_class.tolist(),
        "classified_correct_by_class": classified_correct_by_class.tolist(),
        "cascade_correct_by_class": cascade_correct_by_class.tolist(),
        "confusion_matrix_given_detection": confusion.tolist(),
        "details": details,
    }
    _report_path(output_dir, prefix, "metrics.json").write_text(json.dumps(metrics, indent=2))
    _plot_cascade_bars(metrics, output_dir, prefix)
    overlay_path = _make_overlay_grid(cfg, output_dir, details, samples, backend, prefix)
    return metrics, details if overlay_path else details


def _plot_cascade_bars(metrics: dict, output_dir: Path, prefix: str) -> None:
    labels = [display_class_name(name) for name in metrics["classes"]]
    total = np.array(metrics["total_by_class"])
    detection_correct = np.array(metrics["detection_correct_by_class"])
    classified_total = np.array(metrics["classified_total_by_class"])
    classified_correct = np.array(metrics["classified_correct_by_class"])
    cascade_correct = np.array(metrics["cascade_correct_by_class"])

    x = np.arange(len(labels))
    width = 0.26
    fig, ax = plt.subplots(figsize=(13, 6.5))
    b1 = ax.bar(x - width, detection_correct, width, color="#457b9d", label="Detection correct")
    b2 = ax.bar(x, classified_correct, width, color="#2a9d8f", label="Classified correct after detection")
    b3 = ax.bar(x + width, cascade_correct, width, color="#e76f51", label="End-to-end correct")
    ax.set_title(
        f"GCD validation cascade ({metrics['detector_backend']}): cloud detection -> cloud type classification\n"
        f"det={metrics['detector_image_accuracy']:.1%}, "
        f"cls|det={metrics['classifier_accuracy_given_detection']:.1%}, "
        f"end-to-end={metrics['cascade_accuracy']:.1%}"
    )
    ax.set_ylabel("Images")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha="right")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False)

    for bars, denominators in [(b1, total), (b2, classified_total), (b3, total)]:
        for bar, denom in zip(bars, denominators, strict=False):
            value = int(bar.get_height())
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value + max(total.max() * 0.015, 1),
                f"{value}/{int(denom)}",
                ha="center",
                va="bottom",
                fontsize=8,
                rotation=90,
            )
    fig.tight_layout()
    fig.savefig(_report_path(output_dir, prefix, "bar.png"), dpi=180)
    plt.close(fig)


def _select_samples(details: list[dict], samples: int, seed: int) -> list[dict]:
    rng = random.Random(seed)
    groups = [
        [d for d in details if d["has_detection"] and d["cascade_correct"]],
        [d for d in details if d["has_detection"] and not d["cascade_correct"]],
        [d for d in details if not d["has_detection"] and d["expects_cloud"]],
        [d for d in details if not d["has_detection"] and not d["expects_cloud"]],
    ]
    selected: list[dict] = []
    for group in groups:
        rng.shuffle(group)
        selected.extend(group[: max(1, samples // len(groups))])
    if len(selected) < samples:
        remaining = [d for d in details if d not in selected]
        rng.shuffle(remaining)
        selected.extend(remaining[: samples - len(selected)])
    return selected[:samples]


def _make_overlay_grid(
    cfg: dict,
    output_dir: Path,
    details: list[dict],
    samples: int,
    backend: str,
    prefix: str,
) -> Path | None:
    selected = _select_samples(details, samples, cfg["project"]["seed"])
    if not selected:
        return None

    classes = cfg["data"].get("classification_classes")
    identifier = CloudIdentifier(
        detector_weights=cfg["inference"]["detector_weights"],
        classifier_weights=cfg["inference"]["classifier_weights"],
        class_names=classes,
        detector_backend=backend,
        unet_weights=cfg.get("unet", {}).get("checkpoint"),
        unet_threshold=cfg.get("unet", {}).get("threshold", 0.45),
        unet_min_area=cfg.get("unet", {}).get("min_area", 256),
        device=get_device(cfg),
        image_size=cfg["data"]["image_size"],
        detector_conf=cfg["detector"]["conf"],
        detector_iou=cfg["detector"]["iou"],
        half=cfg["detector"]["half"],
        crop_padding=cfg["inference"]["crop_padding"],
    )

    overlays: list[np.ndarray] = []
    titles: list[str] = []
    for item in tqdm(selected, desc="GCD cascade overlay samples"):
        overlay_bgr, predictions = identifier.predict(item["path"])
        overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)
        if predictions:
            pred = ", ".join(f"{p.class_name} {p.class_confidence:.0%}" for p in predictions[:2])
        else:
            pred = "No cloud detection"
        status = "OK" if item["cascade_correct"] else "FAIL"
        overlays.append(overlay_rgb)
        titles.append(f"{status} | True: {item['true_class']}\nPred: {pred}")

    cols = min(3, len(overlays))
    rows = math.ceil(len(overlays) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5.0 * cols, 4.6 * rows))
    axes_array = np.atleast_1d(axes).reshape(rows, cols)
    for ax in axes_array.ravel():
        ax.axis("off")
    for ax, image, title in zip(axes_array.ravel(), overlays, titles, strict=False):
        ax.imshow(image)
        ax.set_title(title, fontsize=10)
    fig.tight_layout()
    output_path = _report_path(output_dir, prefix, "overlay_samples.jpg")
    fig.savefig(output_path, dpi=180)
    plt.close(fig)
    return output_path


def main() -> None:
    parser = argparse.ArgumentParser(description="Evaluate the GCD detector-classifier cascade.")
    parser.add_argument("--config", default="configs/default.yaml")
    parser.add_argument("--output-dir", default="reports")
    parser.add_argument("--samples", type=int, default=9)
    parser.add_argument("--backend", choices=["yolo", "unet", "hybrid"], default=None)
    parser.add_argument("--prefix", default=None)
    args = parser.parse_args()

    cfg = load_config(args.config)
    seed_everything(cfg["project"]["seed"])
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    backend = args.backend or cfg["detector"].get("backend", "yolo")
    prefix = args.prefix or ("gcd_val_cascade" if args.backend is None else f"gcd_val_{backend}_cascade")

    metrics, _ = _evaluate_cascade(cfg, output_dir, args.samples, backend, prefix)
    print(f"backend={backend}")
    print(f"saved={_report_path(output_dir, prefix, 'bar.png')}")
    print(f"saved={_report_path(output_dir, prefix, 'overlay_samples.jpg')}")
    print(f"saved={_report_path(output_dir, prefix, 'metrics.json')}")
    print(f"GCD detector image accuracy={metrics['detector_image_accuracy']:.4f}")
    print(f"GCD classifier accuracy given detection={metrics['classifier_accuracy_given_detection']:.4f}")
    print(f"GCD cascade accuracy={metrics['cascade_accuracy']:.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile configs/default.yaml
project:
  name: cloud-chaser
  seed: 42
  device: cuda
  output_dir: /kaggle/working/runs

data:
  swimseg_root: /kaggle/input/skyimage-dataset/swimseg-2
  gcd_root: /kaggle/input/gcd/GCD
  prepared_seg_dir: /kaggle/working/swimseg_yolo
  num_workers: 2
  image_size: 224
  val_fraction: 0.15
  seg_val_fraction: 0.1
  seg_test_fraction: 0.1
  classification_classes:
    - 1_cumulus
    - 2_altocumulus
    - 3_cirrus
    - 4_clearsky
    - 5_stratocumulus
    - 6_cumulonimbus
    - 7_mixed
  min_mask_area: 96
  swimseg_invert_masks: false

augmentation:
  random_shadow_p: 0.25
  gaussian_blur_p: 0.20
  hflip_p: 0.50
  vflip_p: 0.15

detector:
  backend: hybrid
  model: yolo11s-seg.pt
  epochs: 120
  imgsz: 640
  batch: 8
  patience: 35
  lr0: 0.01
  weight_decay: 0.0005
  conf: 0.25
  iou: 0.60
  half: true

unet:
  checkpoint: /kaggle/working/runs/unet/best.pt
  epochs: 60
  batch_size: 8
  lr: 0.0003
  weight_decay: 0.00001
  threshold: 0.45
  min_area: 256

classifier:
  backbone: resnet50
  batch_size: 32
  epochs: 80
  lr: 0.0001
  weight_decay: 0.0001
  dropout: 0.2
  amp: true
  freeze_backbone_epochs: 2
  checkpoint: /kaggle/working/runs/classifier/best.pt

inference:
  detector_weights: /kaggle/working/runs/detector/weights/best.pt
  classifier_weights: /kaggle/working/runs/classifier/best.pt
  output_dir: /kaggle/working/predictions
  crop_padding: 12
  mask_alpha: 0.42


## Install Dependencies

In [ ]:
import torch, sys
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


In [ ]:
import subprocess
subprocess.run(['python', '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.'], check=True)


## Configure Dataset Paths

Attach `antigs/skyimage-dataset` and your GCD Kaggle dataset, then run this cell. It auto-discovers common folder names under `/kaggle/input`.

In [ ]:
from pathlib import Path
import yaml
import torch

cfg_path = Path('configs/default.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
KAGGLE_INPUT = Path('/kaggle/input')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def image_count(path):
    return sum(1 for p in Path(path).rglob('*') if p.suffix.lower() in IMAGE_EXTS)

def find_swimseg_root():
    explicit = [
        Path('/kaggle/input/skyimage-dataset/swimseg-2'),
        Path('/kaggle/input/skyimage-dataset/SWIMSEG-2'),
        Path('/kaggle/input/datasets/antigs/skyimage-dataset/swimseg-2'),
        Path('/kaggle/input/datasets/antigs/skyimage-dataset/SWIMSEG-2'),
        Path('/kaggle/input/datasets/skyimage-dataset/swimseg-2'),
    ]
    for path in explicit:
        if path.exists():
            return str(path)
    candidates = []
    if KAGGLE_INPUT.exists():
        for path in KAGGLE_INPUT.rglob('*'):
            if not path.is_dir():
                continue
            if 'swimseg' not in str(path).lower():
                continue
            count = image_count(path)
            if count:
                candidates.append((count, path))
    if candidates:
        return str(sorted(candidates, key=lambda item: (-item[0], len(item[1].parts), str(item[1])))[0][1])
    return ''

def has_split_dir(path, split_names):
    return any((path / name).exists() and (path / name).is_dir() for name in split_names)

def is_not_segmentation_dataset(path):
    text = str(path).lower()
    banned = ['swimseg', 'skyimage', 'sky-image', 'segmentation', 'mask', 'latest-output']
    return not any(token in text for token in banned)

def find_gcd_root():
    explicit = [
        Path('/kaggle/input/gcd/GCD'),
        Path('/kaggle/input/gcd'),
        Path('/kaggle/input/global-cloud-dataset/GCD'),
        Path('/kaggle/input/global-cloud-dataset'),
        Path('/kaggle/input/datasets/alopixalopix/gcd-dataset/GCD'),
        Path('/kaggle/input/datasets/alopixalopix/gcd-dataset'),
    ]
    for path in explicit:
        if path.exists() and is_not_segmentation_dataset(path) and has_split_dir(path, ['train', 'Train', 'training', 'Training']):
            return str(path)
    candidates = []
    if KAGGLE_INPUT.exists():
        for path in KAGGLE_INPUT.rglob('*'):
            if not path.is_dir() or not is_not_segmentation_dataset(path):
                continue
            if has_split_dir(path, ['train', 'Train', 'training', 'Training']):
                class_root = next((path / n for n in ['train', 'Train', 'training', 'Training'] if (path / n).exists()), None)
            else:
                class_root = path
            class_dirs = [p for p in class_root.iterdir() if p.is_dir()] if class_root and class_root.exists() else []
            names = ' '.join(p.name.lower().replace('_', '') for p in class_dirs)
            cloud_hits = sum(1 for token in ['cumulus', 'altocumulus', 'cirrus', 'clearsky', 'stratocumulus', 'cumulonimbus', 'mixed'] if token in names)
            count = image_count(class_root) if class_root else 0
            if len(class_dirs) >= 2 and count and cloud_hits >= 2:
                candidates.append((cloud_hits, count, path))
    if candidates:
        return str(sorted(candidates, key=lambda item: (-item[0], -item[1], len(item[2].parts), str(item[2])))[0][2])
    return ''

cfg['data']['swimseg_root'] = find_swimseg_root()
cfg['data']['gcd_root'] = find_gcd_root()
cfg['data']['prepared_seg_dir'] = '/kaggle/working/swimseg_yolo'
cfg['project']['output_dir'] = '/kaggle/working/runs'
cfg['project']['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg['detector']['backend'] = 'hybrid'
cfg['detector']['epochs'] = 120
cfg['detector']['patience'] = 35
cfg['unet']['epochs'] = 60
cfg['classifier']['epochs'] = 80
cfg['inference']['detector_weights'] = '/kaggle/working/runs/detector/weights/best.pt'
cfg['inference']['classifier_weights'] = '/kaggle/working/runs/classifier/best.pt'
cfg['classifier']['checkpoint'] = '/kaggle/working/runs/classifier/best.pt'
cfg['unet']['checkpoint'] = '/kaggle/working/runs/unet/best.pt'

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg_path.read_text())
for label, key in [('SWIMSEG', 'swimseg_root'), ('GCD', 'gcd_root')]:
    value = cfg['data'][key]
    path = Path(value) if value else Path('__missing__')
    print(f'{label}: {value or "NOT FOUND"} exists={path.exists()}')

if not cfg['data']['swimseg_root'] or not Path(cfg['data']['swimseg_root']).exists():
    print()
    print('Available /kaggle/input folders:')
    for path in sorted(KAGGLE_INPUT.glob('*')) if KAGGLE_INPUT.exists() else []:
        print(' -', path)
    print()
    print('First nested folders:')
    nested = [p for p in sorted(KAGGLE_INPUT.rglob('*')) if p.is_dir()][:120] if KAGGLE_INPUT.exists() else []
    for path in nested:
        print(' -', path)
    raise FileNotFoundError('Attach the Kaggle Sky-Image Dataset, or update data.swimseg_root above.')


In [ ]:
from pathlib import Path
import shutil

LATEST_OUTPUT_ROOT = Path('/kaggle/input/datasets/alopixalopix/latest-output')
RUNS_ROOT = Path('/kaggle/working/runs')
CHECKPOINTS_ROOT = Path('/kaggle/working/checkpoints')

def find_artifact(root, candidates, predicate):
    for candidate in candidates:
        path = root / candidate
        if path.exists():
            return path
    if root.exists():
        matches = [p for p in root.rglob('*') if p.is_file() and predicate(p)]
        if matches:
            return sorted(matches, key=lambda p: (len(p.parts), str(p)))[0]
    return None

def restore_artifact(src, dst):
    if src is None:
        print(f'No saved artifact found for {dst}')
        return False
    if dst.exists():
        print(f'Already present, keeping: {dst}')
        return True
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f'Restored {dst} from {src}')
    return True

def restore_stage(stage, dst_dir):
    for name in ['last.pt', 'best.pt']:
        src = find_artifact(
            LATEST_OUTPUT_ROOT,
            [
                Path(f'runs/{stage}/{name}'),
                Path(f'cloud-chaser/runs/{stage}/{name}'),
                Path(f'{stage}/{name}'),
                Path(f'runs/{stage}/weights/{name}'),
                Path(f'cloud-chaser/runs/{stage}/weights/{name}'),
                Path(f'{stage}/weights/{name}'),
            ],
            lambda p, stage=stage, name=name: p.name == name and stage in str(p).lower(),
        )
        restore_artifact(src, dst_dir / name)

if LATEST_OUTPUT_ROOT.exists():
    print('Restoring saved model artifacts from', LATEST_OUTPUT_ROOT)
    restore_stage('detector', RUNS_ROOT / 'detector/weights')
    restore_stage('classifier', RUNS_ROOT / 'classifier')
    restore_stage('unet', RUNS_ROOT / 'unet')
    torchscript_src = find_artifact(
        LATEST_OUTPUT_ROOT,
        [Path('classifier.torchscript'), Path('artifacts/classifier.torchscript'), Path('cloud-chaser/classifier.torchscript')],
        lambda p: p.name == 'classifier.torchscript',
    )
    restore_artifact(torchscript_src, Path('/kaggle/working/artifacts/classifier.torchscript'))
else:
    print('No latest-output checkpoint dataset attached at', LATEST_OUTPUT_ROOT)

CHECKPOINTS_ROOT.mkdir(parents=True, exist_ok=True)
print('Checkpoint restore directory:', RUNS_ROOT)


## Dataset Tree Debug

Run this if path discovery fails or if the SWIMSEG/GCD folder names differ.

In [ ]:
from pathlib import Path
for root in sorted(Path('/kaggle/input').glob('*')):
    print(root)
    for child in sorted(root.rglob('*'))[:80]:
        print('  ', child.relative_to(root))


## GCD Dataset Sanity Check


In [ ]:
from pathlib import Path
import yaml
cfg = yaml.safe_load(Path('configs/default.yaml').read_text())
gcd_value = cfg['data'].get('gcd_root')
print('GCD root:', gcd_value or 'NOT FOUND')
if not gcd_value:
    print('No GCD classification dataset was detected. Attach GCD or set data.gcd_root manually.')
else:
    gcd_root = Path(gcd_value)
    print('exists=', gcd_root.exists())
    for split_name in ['train', 'Train', 'training', 'Training', 'test', 'Test', 'val', 'validation']:
        split_dir = gcd_root / split_name
        if split_dir.exists():
            print('\n', split_dir)
            for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
                count = sum(1 for p in class_dir.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'})
                print(' ', class_dir.name, count)


## Prepare SWIMSEG-2 for YOLO Segmentation

In [ ]:
import subprocess
subprocess.run(['python', '-m', 'scripts.prepare_swimseg_yolo', '--config', 'configs/default.yaml'], check=True)


## Train Detector

In [ ]:
from pathlib import Path
import subprocess

assert Path('train.py').exists(), 'train.py was not written; rerun notebook from the top.'
print('Training/extending YOLO detector. Existing last.pt will be resumed automatically if present.')
subprocess.run(['python', 'train.py', 'detector', '--config', 'configs/default.yaml'], check=True)


## Train U-Net Fallback Detector


In [ ]:
from pathlib import Path
import subprocess

assert Path('train.py').exists(), 'train.py was not written; rerun notebook from the top.'
print('Training/extending U-Net fallback detector. Existing last.pt will be resumed automatically if present.')
subprocess.run(['python', 'train.py', 'unet', '--config', 'configs/default.yaml'], check=True)


## Train GCD Classifier

In [ ]:
from pathlib import Path
import subprocess
import yaml

cfg = yaml.safe_load(Path('configs/default.yaml').read_text())
print('Configured GCD root:', cfg['data'].get('gcd_root') or 'NOT FOUND')
gcd_root = cfg['data'].get('gcd_root')
if not gcd_root or not Path(gcd_root).exists():
    print('Skipping classifier training: no GCD classification dataset is attached.')
else:
    print('Training/extending classifier. Existing last.pt will be resumed automatically if present.')
    subprocess.run(['python', 'train.py', 'classifier', '--config', 'configs/default.yaml'], check=True)


## Evaluate

## Snapshot Checkpoints


In [ ]:
from pathlib import Path
import shutil

CHECKPOINTS = Path('/kaggle/working/checkpoints')
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
items = {
    'detector_best.pt': Path('/kaggle/working/runs/detector/weights/best.pt'),
    'detector_last.pt': Path('/kaggle/working/runs/detector/weights/last.pt'),
    'unet_best.pt': Path('/kaggle/working/runs/unet/best.pt'),
    'unet_last.pt': Path('/kaggle/working/runs/unet/last.pt'),
    'classifier_best.pt': Path('/kaggle/working/runs/classifier/best.pt'),
    'classifier_last.pt': Path('/kaggle/working/runs/classifier/last.pt'),
}
for name, src in items.items():
    if src.exists():
        dst = CHECKPOINTS / name
        shutil.copy2(src, dst)
        print('checkpoint=', dst)
    else:
        print('missing checkpoint source=', src)


In [ ]:
from pathlib import Path
import subprocess

best_detector = Path('/kaggle/working/runs/detector/weights/best.pt')
best_classifier = Path('/kaggle/working/runs/classifier/best.pt')
best_unet = Path('/kaggle/working/runs/unet/best.pt')

if best_detector.exists():
    print(f'Evaluating detector checkpoint: {best_detector}')
    result = subprocess.run(['python', 'train.py', 'eval-detector', '--config', 'configs/default.yaml'], check=False)
    if result.returncode:
        print('Detector evaluation failed, but the detector checkpoint exists. Continuing notebook run.')
else:
    print('Skipping detector evaluation: missing', best_detector)


if best_unet.exists():
    print(f'Evaluating U-Net checkpoint: {best_unet}')
    result = subprocess.run(['python', 'train.py', 'eval-unet', '--config', 'configs/default.yaml'], check=False)
    if result.returncode:
        print('U-Net evaluation failed, but the U-Net checkpoint exists. Continuing notebook run.')
else:
    print('Skipping U-Net evaluation: missing', best_unet)

if best_classifier.exists():
    print(f'Evaluating classifier checkpoint: {best_classifier}')
    result = subprocess.run(['python', 'train.py', 'eval-classifier', '--config', 'configs/default.yaml'], check=False)
    if result.returncode:
        print('Classifier evaluation failed, but the classifier checkpoint exists. Continuing notebook run.')
else:
    print('Skipping classifier evaluation: missing', best_classifier)


## GCD Backend Experiment Report

Evaluates the same GCD validation cascade with three detector backends: YOLO-only, U-Net-only, and hybrid YOLO-to-U-Net fallback. This cell writes separate per-backend reports plus a comparison chart so we can measure accuracy and runtime tradeoffs directly.


In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import matplotlib.pyplot as plt
from IPython.display import Image, display

REPORT_DIR = Path('/kaggle/working/reports')
REPORT_DIR.mkdir(parents=True, exist_ok=True)
BACKENDS = ['yolo', 'unet', 'hybrid']
SAMPLES = '9'

metrics_rows = []
for backend in BACKENDS:
    prefix = f'gcd_val_{backend}_cascade'
    metrics_path = REPORT_DIR / f'{prefix}_metrics.json'
    print(f'\n=== Evaluating {backend} backend ===')
    subprocess.run([
        'python',
        'scripts/gcd_visual_report.py',
        '--config',
        'configs/default.yaml',
        '--output-dir',
        str(REPORT_DIR),
        '--samples',
        SAMPLES,
        '--backend',
        backend,
        '--prefix',
        prefix,
    ], check=True)
    metrics = json.loads(metrics_path.read_text())
    metrics_rows.append({
        'backend': backend,
        'detector_image_accuracy': metrics['detector_image_accuracy'],
        'classifier_accuracy_given_detection': metrics['classifier_accuracy_given_detection'],
        'cascade_accuracy': metrics['cascade_accuracy'],
        'num_images': metrics['num_images'],
        'detected_images': sum(metrics['detected_by_class']),
    })

summary_path = REPORT_DIR / 'gcd_backend_experiment_summary.csv'
with summary_path.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(metrics_rows[0].keys()))
    writer.writeheader()
    writer.writerows(metrics_rows)
print('\nBackend comparison:')
for row in metrics_rows:
    print(
        f"{row['backend']:>6} | "
        f"det={row['detector_image_accuracy']:.1%} | "
        f"cls|det={row['classifier_accuracy_given_detection']:.1%} | "
        f"cascade={row['cascade_accuracy']:.1%} | "
        f"detected={row['detected_images']}/{row['num_images']}"
    )
print('saved=', summary_path)

fig, ax = plt.subplots(figsize=(9, 5))
x = list(range(len(metrics_rows)))
width = 0.25
det_vals = [r['detector_image_accuracy'] for r in metrics_rows]
cls_vals = [r['classifier_accuracy_given_detection'] for r in metrics_rows]
cas_vals = [r['cascade_accuracy'] for r in metrics_rows]
labels = [r['backend'] for r in metrics_rows]
ax.bar([i - width for i in x], det_vals, width, label='Detection gate')
ax.bar(x, cls_vals, width, label='Classifier | detection')
ax.bar([i + width for i in x], cas_vals, width, label='End-to-end cascade')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1)
ax.set_ylabel('Accuracy')
ax.set_title('GCD validation backend experiment')
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=False)
for container in ax.containers:
    ax.bar_label(container, labels=[f'{v:.1%}' for v in container.datavalues], fontsize=8, padding=2)
fig.tight_layout()
comparison_path = REPORT_DIR / 'gcd_backend_experiment_comparison.png'
fig.savefig(comparison_path, dpi=180)
plt.close(fig)
print('saved=', comparison_path)
display(Image(filename=str(comparison_path)))

for backend in BACKENDS:
    prefix = f'gcd_val_{backend}_cascade'
    for image_path in [REPORT_DIR / f'{prefix}_bar.png', REPORT_DIR / f'{prefix}_overlay_samples.jpg']:
        if image_path.exists():
            print(image_path)
            display(Image(filename=str(image_path)))
        else:
            print('Missing report image:', image_path)


## Inference Example

Change `IMAGE_PATH` to a real image in `/kaggle/input` or `/kaggle/working`.

In [ ]:
from pathlib import Path
import subprocess
import yaml
from IPython.display import Image, display

IMAGE_PATH = Path('/kaggle/input/datasets/alopixalopix/examples/cirocumulus.jpg')
OUTPUT_PATH = Path('/kaggle/working/prediction.jpg')
DETECTOR_BEST = Path('/kaggle/working/runs/detector/weights/best.pt')
CLASSIFIER_BEST = Path('/kaggle/working/runs/classifier/best.pt')
CLASSIFIER_TS = Path('/kaggle/working/artifacts/classifier.torchscript')

classifier_artifact = CLASSIFIER_TS if CLASSIFIER_TS.exists() else CLASSIFIER_BEST
required = [DETECTOR_BEST, classifier_artifact, IMAGE_PATH]
missing = [str(p) for p in required if not p.exists()]
if missing:
    print('Skipping two-stage inference because required files are missing:', missing)
else:
    cfg_path = Path('configs/default.yaml')
    cfg = yaml.safe_load(cfg_path.read_text())
    cfg['inference']['detector_weights'] = str(DETECTOR_BEST)
    cfg['inference']['classifier_weights'] = str(classifier_artifact)
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print('Using classifier artifact:', classifier_artifact)
    subprocess.run(['python', 'inference.py', '--config', 'configs/default.yaml', '--image', str(IMAGE_PATH), '--output', str(OUTPUT_PATH)], check=True)
    display(Image(filename=str(OUTPUT_PATH)))


## Export Artifacts

In [ ]:
from pathlib import Path
import subprocess
import sys

ARTIFACTS = Path('/kaggle/working/artifacts')
ARTIFACTS.mkdir(parents=True, exist_ok=True)
best_detector = Path('/kaggle/working/runs/detector/weights/best.pt')
best_classifier = Path('/kaggle/working/runs/classifier/best.pt')
detector_onnx = Path('/kaggle/working/runs/detector/weights/best.onnx')
classifier_ts = ARTIFACTS / 'classifier.torchscript'

if best_detector.exists():
    if detector_onnx.exists():
        print('Skipping detector ONNX export; found', detector_onnx)
    else:
        print('Exporting detector ONNX quietly...')
        with open('/kaggle/working/detector_export.log', 'w') as log:
            subprocess.run(
                ['python', 'export.py', 'detector', '--config', 'configs/default.yaml', '--format', 'onnx'],
                stdout=log,
                stderr=subprocess.STDOUT,
                check=False,
            )
        print('Detector export log: /kaggle/working/detector_export.log')
else:
    print('Skipping detector export: missing', best_detector)

if best_classifier.exists():
    print('Skipping classifier ONNX export to avoid optional onnxscript noise. TorchScript is the supported classifier export here.')
    if classifier_ts.exists():
        print('Skipping classifier TorchScript export; found', classifier_ts)
    else:
        result = subprocess.run(
            ['python', 'export.py', 'classifier', '--config', 'configs/default.yaml', '--format', 'torchscript', '--output', str(classifier_ts)],
            check=False,
        )
        if result.returncode:
            print('Classifier TorchScript export failed; continuing because best.pt is available.')
else:
    print('Skipping classifier export: missing', best_classifier)
